# Vocacional Bayesiano — Proyecto MIA (UDLA)
**Plataforma de orientación vocacional con redes bayesianas · Alliance Academy International (DECE)**

Este notebook se construye **por módulos**. Cada módulo termina con una **lista de aprobación**. No se avanza al siguiente hasta aprobar el anterior.

| # | Módulo | Estado |
|---|---|---|
| 0 | Propósito, principios y arquitectura | ✅ aprobado |
| 1 | Espacio canónico, contrato de fuentes y ponderación explicable | ✅ aprobado |
| 2 | Estructura de datos en Drive y carga inicial | ✅ aprobado |
| 3 | Limpieza de calificaciones → rendimiento por área | 🟡 en revisión |
| 3B | Simulación CHASIDE y escenarios de estrés (solo pruebas) | 🟡 en revisión |
| 4 | Adaptador CHASIDE (núcleo) y fuentes complementarias | 🟡 en revisión |
| 5 | Modelo bayesiano (Interés × Rendimiento) | 🟡 en revisión |
| 6 | Plano cartesiano vocacional | 🔗 en la interfaz (datos en el JSON del Módulo 9) |
| 7 | Carreras: afinidad, P(top 5) y razones | 🟡 en revisión |
| 8 | Evaluación del modelo | 🟡 incluida en 5.E, 5.F y 7.D |
| 9 | Exportación para la interfaz (JSON) | 🟡 en revisión |

---
# MÓDULO 0 · Propósito, principios y arquitectura

## 0.1 Objetivo
Estimar, para cada estudiante y para cada área vocacional, **dos magnitudes con su incertidumbre**:

- **Interés:** cuánto le atrae el área. Fuentes: pruebas vocacionales.
- **Rendimiento:** qué tan bien se desempeña en lo relacionado con el área. Fuentes: calificaciones y aptitud autopercibida.

Con esas dos magnitudes el sistema ubica cada área en un **plano cartesiano** y propone **carreras afines**. Cada número que aparece debe poder explicarse con una fórmula y un criterio declarado.

## 0.2 Principios de diseño
1. **Explicabilidad antes que exactitud.** Cada peso se deriva de una fórmula y de criterios escritos. No se fijan "a ojo".
2. **Núcleo obligatorio: Notas + CHASIDE.** El sistema funciona completo solo con estas dos fuentes. Todo lo demás (IPP-J, otras pruebas, entrevistas…) es **complementario y opcional**: se traduce al espacio CHASIDE y *suma* evidencia que reduce la incertidumbre, pero nunca es requisito.
3. **Más fuentes = más exactitud, no más ruido.** Cada fuente aporta según su confiabilidad. Si dos fuentes miden lo mismo con el mismo método, no se cuentan dos veces.
4. **Ausencia ≠ cero.** Si una fuente no dice nada sobre un área, esa área queda *sin evidencia* (NaN). Nunca se registra como valor bajo.
5. **El modelo apoya, no decide.** La salida es información para el DECE, docentes y familias.
6. **Componentes reemplazables.** Las pruebas (adaptadores) y las carreras (referencia por país) son configuración, no código.

## 0.3 Arquitectura
```
 FUENTES (n, abiertas)           MÓDULO 1-4                  MÓDULO 5              MÓDULO 6-7
 ┌───────────────┐   adaptador   ┌──────────────────┐        ┌─────────────┐      ┌──────────────────┐
 │ Calificaciones│──────────────▶│                  │        │  Posterior  │      │ Plano cartesiano │
 ├───────────────┤               │  ESPACIO         │ evid.  │  Interés    │─────▶│ Interés×Rendim.  │
 │ CHASIDE       │──────────────▶│  CANÓNICO        │───────▶│  N(μ, σ²)   │      ├──────────────────┤
 ╞═══════════════╡ ◀ NÚCLEO      │  CHASIDE (7 ejes)│ + σ²   │  Rendimiento│─────▶│ Carreras afines  │
 │ IPP-J (opc.)  │┄┄┄┄┄┄┄┄┄┄┄┄┄┄▶│  × 2 dimensiones │        │  N(μ, σ²)   │      │ (tabla por país) │
 ├───────────────┤               │                  │        └─────────────┘      └──────────────────┘
 │ otra (opc.)   │┄┄┄┄┄┄┄┄┄┄┄┄┄┄▶│                  │               ▲
 └───────────────┘               └──────────────────┘               │
                                    rúbrica de confiabilidad ───────┘  (define cuánto pesa cada fuente)
```

In [ ]:
# ── 0.4 Configuración del entorno ───────────────────────────────────────
# Solo librerías estándar de Colab (sin costos de licencia, según la restricción del proyecto).
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from scipy.stats import norm

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)
np.random.seed(42)          # reproducibilidad de cualquier simulación

print('Entorno listo · numpy', np.__version__, '· pandas', pd.__version__)

### ✅ Aprobación Módulo 0
- [ ] El objetivo (Interés y Rendimiento por área, con incertidumbre) representa lo que queremos.
- [ ] Los 6 principios están de acuerdo con la tesis.
- [ ] La arquitectura (fuentes → canónico → posterior → plano/carreras) es la correcta.

---
# MÓDULO 1 · Espacio canónico, contrato de fuentes y ponderación explicable

**Qué resuelve este módulo:** define *el idioma común* en el que todas las fuentes se expresan y *la regla* que decide cuánto pesa cada una.

## 1.1 Espacio canónico: CHASIDE

**¿RIASEC es lo mismo que CHASIDE?** No. Son dos marcos distintos que se superponen:

| | RIASEC (Holland) | CHASIDE |
|---|---|---|
| Qué clasifica | Tipos de personalidad vocacional | Áreas de carreras |
| Dimensiones | 6: Realista, Investigador, Artístico, Social, Emprendedor, Convencional | 7: Administrativas, Humanísticas, Artísticas, Salud, Técnicas/Ingenierías, Defensa, Exactas/Agrarias |
| Uso | Internacional (O*NET, EE. UU.) | Iberoamérica; muy usado en DECE de Ecuador |
| Mide | Intereses | **Intereses (10 ítems por área) y aptitudes (4 ítems por área)** |

**Decisión:** el espacio canónico es **CHASIDE**. Hay dos razones. Es el instrumento más cercano a la práctica del DECE en Ecuador. Y ya separa *interés* y *aptitud*, que son justo los dos ejes del plano cartesiano.

RIASEC, IPP-J o cualquier otra prueba entran mediante una **matriz de equivalencia** hacia CHASIDE (sección 1.3).

In [ ]:
# ── 1.1 Definición del espacio canónico ────────────────────────────────
EJES = {
    'C': 'Administrativas y Contables',
    'H': 'Humanísticas y Ciencias Sociales',
    'A': 'Artísticas',
    'S': 'Ciencias de la Salud',
    'I': 'Ingenierías, Técnicas y Computación',
    'D': 'Defensa y Seguridad',
    'E': 'Ciencias Exactas y Agrarias',
}
CODIGOS = list(EJES.keys())          # orden fijo: C H A S I D E

# Cada área se describe con DOS dimensiones latentes (no observables directamente):
DIMENSIONES = {
    'interes':     'Cuánto le atrae el área (fuentes: pruebas de intereses)',
    'rendimiento': 'Qué tan bien se desempeña en lo relacionado al área (fuentes: notas, aptitud autopercibida)',
}

pd.DataFrame({'Área': EJES}).rename_axis('Código')

## 1.2 Escala común: puntuación z interna, percentil para mostrar

Cada fuente usa su propia escala: CHASIDE de 0 a 10 por área, notas de 0 a 100 o de 0 a 4, IPP-J con otra distinta. Para combinarlas hay que llevarlas a la misma unidad.

**Criterio:** internamente todo se expresa en **puntuación z respecto de la cohorte**:

$$z = \frac{x - \bar{x}_{cohorte}}{s_{cohorte}}$$

- **z = 0** es el promedio del grupo. **z = +1** está una desviación estándar por encima.
- **Por qué z y no 0–5:** z elimina las diferencias de escala *y* de dificultad. Un 8/10 en una prueba fácil no equivale a un 8/10 en una difícil. Además, el modelo gaussiano del Módulo 5 trabaja de forma natural en esta escala.
- **Para mostrar** (docentes y padres) se convierte a **percentil**: $P = \Phi(z) \times 100$. "Está por encima del 84 % de su grupo" se entiende mejor que "z = 1".

In [ ]:
# ── 1.2 Funciones de escala ────────────────────────────────────────────
def a_z(valores, media=None, sd=None):
    # Estandariza respecto de la cohorte. Si se pasan media/sd (normas publicadas), se usan esas.
    v = pd.Series(valores, dtype=float)
    media = v.mean() if media is None else media
    sd = v.std(ddof=1) if sd is None else sd
    return (v - media) / sd

def z_a_percentil(z):
    return norm.cdf(z) * 100

# Ejemplo: misma persona, dos escalas distintas → mismo lenguaje
notas_0_100 = [70, 80, 90, 85, 95]
chaside_0_10 = [3, 5, 8, 6, 9]
pd.DataFrame({
    'nota_0_100': notas_0_100, 'z_nota': a_z(notas_0_100).round(2),
    'chaside_0_10': chaside_0_10, 'z_chaside': a_z(chaside_0_10).round(2),
    'percentil_chaside': z_a_percentil(a_z(chaside_0_10)).round(0),
})

## 1.3 Contrato de fuente: cómo entra una prueba nueva

**Respuesta a "¿cada fuente nueva crea otra estructura?"** No. Cada fuente se declara con una **ficha** (un adaptador) que contiene:

| Campo | Para qué |
|---|---|
| `tipo` | Si aporta a **interés** o a **rendimiento** |
| `ejes_origen` | Las escalas propias de la prueba |
| `equivalencia` | Matriz **M**: cuánto aporta cada escala de origen a cada área CHASIDE |
| `alfa` | Confiabilidad del instrumento (α de Cronbach u otra) |
| `familia` | Método de medición; sirve para no contar dos veces la misma evidencia |
| `criterios` | Criterios cualitativos de calidad (sección 1.4) |

**Proyección al espacio canónico.** Para cada área $c$ se calcula el promedio ponderado de las escalas de origen que le corresponden:

$$x_c = \frac{\sum_j M_{c,j}\, x_j}{\sum_j M_{c,j}} \qquad \text{si } \sum_j M_{c,j} = 0 \Rightarrow x_c = \text{NaN (sin evidencia)}$$

Así una prueba con 15 campos profesionales, o una con 6 tipos RIASEC, termina hablando en las 7 áreas CHASIDE. Las áreas que la prueba no cubre quedan **vacías, no en cero**.

In [ ]:
# ── 1.3 Contrato de fuente ─────────────────────────────────────────────
@dataclass
class Fuente:
    id: str
    nombre: str
    tipo: str                      # 'interes' | 'rendimiento'
    ejes_origen: list
    equivalencia: dict             # {eje_origen: {eje_CHASIDE: peso}}
    alfa: float                    # confiabilidad publicada o calculada (0-1)
    familia: str                   # método de medición (para correlación entre fuentes)
    criterios: dict = field(default_factory=dict)   # criterios cualitativos FIJOS de la fuente
    nucleo: bool = False           # True = obligatoria (Notas, CHASIDE). False = complementaria opcional
    nota: str = ''

    def matriz(self):
        # Matriz M (filas = áreas CHASIDE, columnas = ejes de origen)
        M = pd.DataFrame(0.0, index=CODIGOS, columns=self.ejes_origen)
        for origen, destinos in self.equivalencia.items():
            for c, w in destinos.items():
                M.loc[c, origen] = w
        return M

    def cobertura(self):
        # Qué áreas CHASIDE recibe evidencia de esta fuente
        return (self.matriz().sum(axis=1) > 0)

    def validar(self):
        errores = []
        if self.tipo not in DIMENSIONES: errores.append(f'tipo inválido: {self.tipo}')
        if not 0 < self.alfa < 1:        errores.append('alfa debe estar entre 0 y 1')
        for o, d in self.equivalencia.items():
            if o not in self.ejes_origen: errores.append(f'eje de origen desconocido: {o}')
            for c in d:
                if c not in CODIGOS:      errores.append(f'área CHASIDE desconocida: {c}')
        return errores or ['OK']

def verificar_nucleo(fuentes_aplicadas):
    # El sistema exige el núcleo (Notas + CHASIDE); las complementarias son opcionales.
    presentes = {f.id for f, _ in fuentes_aplicadas}
    faltan = [f.nombre for f in FUENTES if f.nucleo and f.id not in presentes]
    if faltan:
        raise ValueError(f'Falta evidencia del núcleo: {faltan}. El sistema no puede estimar sin Notas + CHASIDE.')
    extra = [f.nombre for f, _ in fuentes_aplicadas if not f.nucleo]
    return 'Núcleo completo' + (f' + complementarias: {extra}' if extra else ' (sin complementarias)')

def proyectar(fuente, puntajes_origen):
    # Lleva un vector de puntajes (en z) de la fuente al espacio CHASIDE. NaN = sin evidencia.
    M = fuente.matriz()
    x = pd.Series(puntajes_origen, dtype=float).reindex(fuente.ejes_origen)
    num = M.mul(x, axis=1).sum(axis=1, min_count=1)
    den = M.loc[:, x.notna()].sum(axis=1)
    return (num / den.replace(0, np.nan)).rename(fuente.id)

## 1.4 Rúbrica de confiabilidad: de criterios cualitativos a un peso numérico

Aquí entran los **criterios no numéricos** que también deben pesar. El procedimiento tiene tres pasos.

**Paso 1 · Varianza de error base, a partir de la confiabilidad α.**
En teoría clásica de tests, *observado = verdadero + error*. Si la varianza verdadera se fija en 1 (escala z), la varianza del error es:

$$\sigma^2_{base} = \frac{1-\alpha}{\alpha}$$

Por ejemplo, α = 0.90 da σ² = 0.11 (fuente muy precisa), y α = 0.60 da σ² = 0.67 (fuente ruidosa).

**Paso 2 · Factores cualitativos.** Cada criterio cualitativo **multiplica** la varianza por un factor ≥ 1. Un factor mayor significa más ruido y, por tanto, menos peso. Hay criterios de dos tipos:

- **De la fuente** (fijos): validación del instrumento y calidad de la equivalencia con CHASIDE.
- **De cada aplicación** (por estudiante): antigüedad, completitud, modalidad y consistencia.

$$\sigma^2_{ef} = \sigma^2_{base} \times \prod_k f_k$$

**Paso 3 · Peso = precisión normalizada.** La precisión es $\tau = 1/\sigma^2_{ef}$. El peso de cada fuente es:

$$w_i = \frac{\tau_i}{\tau_{prior} + \sum_j \tau_j}$$

Esta fórmula no es arbitraria: es exactamente la media posterior del modelo bayesiano gaussiano del Módulo 5. **Los porcentajes no se eligen; son consecuencia de la confiabilidad.**

> ⚠️ Los valores de los factores son **supuestos expertos documentados**. En el Módulo 8 se hará un *análisis de sensibilidad* para mostrar cuánto cambian las recomendaciones si se modifican. Así se cumple lo que pide la ISO/IEC 25059 en transparencia.

In [ ]:
# ── 1.4 Rúbrica de criterios cualitativos ──────────────────────────────
# Cada criterio: nivel → (factor multiplicador de varianza, justificación)
RUBRICA = {
    # --- criterios de la FUENTE ---
    'validacion': {
        'publicada':      (1.0, 'Instrumento con validación psicométrica publicada'),
        'adaptacion':     (1.3, 'Adaptación local sin validación formal'),
        'ad_hoc':         (2.0, 'Instrumento propio sin validación'),
    },
    'equivalencia': {
        'directa':        (1.0, 'La prueba mide las mismas áreas CHASIDE'),
        'aproximada':     (1.5, 'Escalas distintas mapeadas por tabla de equivalencia'),
        'debil':          (2.5, 'Relación indirecta (p. ej. notas → área)'),
    },
    # --- criterios de cada APLICACIÓN (por estudiante) ---
    'antiguedad': {
        'menos_1_anio':   (1.0, 'Aplicada en el último año'),
        '1_a_2_anios':    (1.2, 'Los intereses adolescentes cambian; evidencia algo desactualizada'),
        'mas_2_anios':    (1.5, 'Evidencia antigua'),
    },
    'completitud': {
        'completa':       (1.0, '≥ 95 % de ítems respondidos'),
        'parcial':        (1.3, '80–95 % de ítems respondidos'),
        'incompleta':     (2.0, '< 80 % de ítems respondidos'),
    },
    'modalidad': {
        'supervisada':    (1.0, 'Aplicada por el DECE'),
        'autonoma':       (1.2, 'En línea sin supervisión'),
    },
    'consistencia': {
        'sin_alertas':    (1.0, 'Patrón de respuesta normal'),
        'alerta':         (2.0, 'Respuestas aleatorias o en patrón (todo sí / todo no)'),
    },
}

# Correlación de errores entre fuentes de la MISMA familia (mismo método de medición).
# Supuesto: dos autoinformes de intereses comparten sesgos (deseabilidad social, ánimo del día).
RHO_FAMILIA = {'autoinforme_intereses': 0.30, 'autoinforme_aptitud': 0.30, 'desempeno_academico': 0.0}

TAU_PRIOR = 1.0   # prior N(0,1): sin evidencia, el estudiante está en el promedio de su cohorte

pd.DataFrame([(c, n, f, j) for c, niv in RUBRICA.items() for n, (f, j) in niv.items()],
             columns=['criterio', 'nivel', 'factor', 'justificación'])

In [ ]:
# ── 1.5 Motor de ponderación explicable ────────────────────────────────
def sigma2_efectiva(fuente, criterios_aplicacion=None, n_misma_familia=1):
    crit = {**fuente.criterios, **(criterios_aplicacion or {})}
    s2_base = (1 - fuente.alfa) / fuente.alfa
    detalle, producto = [], 1.0
    for c, nivel in crit.items():
        f, just = RUBRICA[c][nivel]
        producto *= f
        if f != 1.0: detalle.append(f'{c}={nivel} (×{f})')
    # Corrección por fuentes correlacionadas: con k fuentes de la misma familia y correlación ρ,
    # la información efectiva se reduce en (1 + (k-1)ρ)  → se infla la varianza en ese factor.
    rho = RHO_FAMILIA.get(fuente.familia, 0.0)
    f_corr = 1 + (n_misma_familia - 1) * rho
    if f_corr != 1.0: detalle.append(f'misma familia k={n_misma_familia} (×{f_corr:.2f})')
    return s2_base, s2_base * producto * f_corr, detalle

def explicar_pesos(fuentes_aplicadas, dimension):
    # fuentes_aplicadas: lista de (Fuente, criterios_aplicacion). Devuelve tabla + explicación en texto.
    usadas = [(f, c) for f, c in fuentes_aplicadas if f.tipo == dimension]
    conteo = pd.Series([f.familia for f, _ in usadas]).value_counts()
    filas = [{'fuente': 'Prior (promedio cohorte)', 'alfa': None, 's2_base': None,
              'ajustes': 'punto de partida sin evidencia', 's2_efectiva': 1 / TAU_PRIOR, 'precision': TAU_PRIOR}]
    for f, c in usadas:
        s2b, s2e, det = sigma2_efectiva(f, c, conteo[f.familia])
        filas.append({'fuente': f.nombre, 'alfa': f.alfa, 's2_base': round(s2b, 3),
                      'ajustes': ', '.join(det) or '—', 's2_efectiva': round(s2e, 3), 'precision': round(1 / s2e, 3)})
    t = pd.DataFrame(filas)
    t['peso_%'] = (100 * t['precision'] / t['precision'].sum()).round(1)
    return t

def narrar(tabla, dimension):
    print(f'▶ Ponderación para la dimensión «{dimension.upper()}»')
    for _, r in tabla.iloc[1:].iterrows():
        print(f"  • {r['fuente']}: pesa {r['peso_%']} %. Confiabilidad α={r['alfa']} → σ² base {r['s2_base']}; "
              f"ajustes: {r['ajustes']} → σ² efectiva {r['s2_efectiva']}.")
    print(f"  • El prior (promedio del grupo) conserva {tabla.iloc[0]['peso_%']} %: es lo que el modelo asume sin evidencia.")
    sd_post = 1 / np.sqrt(tabla['precision'].sum())
    print(f"  ⇒ Incertidumbre final σ = 1/√(Σ precisiones) = {sd_post:.3f}  (sin evidencia sería 1.000)")

## 1.6 Demostración con las fuentes actuales

> ⚠️ **Valores de α:** son **supuestos provisionales** marcados como tales. En el Módulo 4 el α de CHASIDE se **calcula** con las respuestas por ítem que ya tenemos (108 columnas en el archivo del test). La ficha del IPP-J es **ilustrativa**: se reemplaza con su estructura real cuando el DECE la entregue.

Fíjate en tres detalles:

- **CHASIDE-aptitud** tiene solo 4 ítems por área, así que su α es menor y pesa menos.
- **Calificaciones** tiene equivalencia "débil", porque la tabla materia→área es indirecta.
- **IPP-J es complementaria:** si está, suma precisión y reduce la incertidumbre. Si no está, el sistema funciona igual con el núcleo. Cuando coexiste con CHASIDE-intereses, ambas se penalizan por pertenecer a la misma familia.

In [ ]:
# ── 1.6 Registro de fuentes ────────────────────────────────────────────
identidad = {c: {c: 1.0} for c in CODIGOS}

CHASIDE_INT = Fuente('chaside_int', 'CHASIDE · intereses', 'interes', CODIGOS, identidad,
                     alfa=0.80, familia='autoinforme_intereses',
                     criterios={'validacion': 'publicada', 'equivalencia': 'directa'}, nucleo=True,
                     nota='10 ítems por área. α PROVISIONAL → se calcula en Módulo 4.')

CHASIDE_APT = Fuente('chaside_apt', 'CHASIDE · aptitudes', 'rendimiento', CODIGOS, identidad,
                     alfa=0.60, familia='autoinforme_aptitud',
                     criterios={'validacion': 'publicada', 'equivalencia': 'directa'}, nucleo=True,
                     nota='Solo 4 ítems por área → menor confiabilidad. α PROVISIONAL.')

NOTAS = Fuente('notas', 'Calificaciones (grados 9-12)', 'rendimiento', CODIGOS, identidad,
               alfa=0.85, familia='desempeno_academico',
               criterios={'validacion': 'publicada', 'equivalencia': 'debil'}, nucleo=True,
               nota='Promedio de muchas materias (alta consistencia), pero relación indirecta con el área.')

# IPP-J ILUSTRATIVO: campos de ejemplo. Reemplazar con la estructura real del instrumento del DECE.
IPPJ_DEMO = Fuente('ippj', 'IPP-J (ilustrativo)', 'interes',
                   ['Cientifico', 'Sanitario', 'Artistico', 'Juridico', 'Economico', 'Tecnico', 'Seguridad'],
                   {'Cientifico': {'E': 1.0, 'I': 0.3}, 'Sanitario': {'S': 1.0},
                    'Artistico': {'A': 1.0}, 'Juridico': {'H': 1.0, 'D': 0.3},
                    'Economico': {'C': 1.0}, 'Tecnico': {'I': 1.0}, 'Seguridad': {'D': 1.0}},
                   alfa=0.85, familia='autoinforme_intereses',
                   criterios={'validacion': 'publicada', 'equivalencia': 'aproximada'}, nucleo=False,
                   nota='COMPLEMENTARIA y opcional. DEMO: estructura real pendiente.')

FUENTES = [CHASIDE_INT, CHASIDE_APT, NOTAS, IPPJ_DEMO]
for f in FUENTES:
    print(f'{"NÚCLEO" if f.nucleo else "opcional":8s} {f.nombre:30s} validación: {f.validar()[0]:4s} · cobertura: {"".join(c if v else "·" for c, v in f.cobertura().items())}')

print('\nMatriz de equivalencia IPP-J (demo) → CHASIDE:')
IPPJ_DEMO.matriz()

In [ ]:
# ── 1.7 Ponderación para tres estudiantes ──────────────────────────────
OK = {'antiguedad': 'menos_1_anio', 'completitud': 'completa', 'modalidad': 'supervisada', 'consistencia': 'sin_alertas'}

# A · Solo NÚCLEO (Notas + CHASIDE), todo reciente y completo → caso mínimo con el que el sistema debe funcionar.
est_A = [(CHASIDE_INT, OK), (CHASIDE_APT, OK), (NOTAS, {})]
# B · NÚCLEO + IPP-J complementaria → misma calidad, más evidencia: debe bajar la incertidumbre de INTERÉS.
est_B = est_A + [(IPPJ_DEMO, OK)]
# C · Solo NÚCLEO pero CHASIDE de hace 2 años y en línea → la evidencia vale menos.
VIEJO = {**OK, 'antiguedad': 'mas_2_anios', 'modalidad': 'autonoma'}
est_C = [(CHASIDE_INT, VIEJO), (CHASIDE_APT, VIEJO), (NOTAS, {})]

for nombre, est in [('A · núcleo', est_A), ('B · núcleo + IPP-J', est_B), ('C · núcleo con CHASIDE antiguo', est_C)]:
    print('═' * 100, f'\nESTUDIANTE {nombre} → {verificar_nucleo(est)}')
    for dim in DIMENSIONES:
        t = explicar_pesos(est, dim)
        display(t)
        narrar(t, dim)
        print()

# Sin núcleo → el sistema se niega a estimar (por diseño)
try:
    verificar_nucleo([(IPPJ_DEMO, OK)])
except ValueError as e:
    print('⛔', e)

**Cómo leer el resultado.**

- **A → B:** agregar el IPP-J (opcional) **reduce la incertidumbre** del interés. Eso es "más datos = más exactitud", medido.
- **A → C:** las mismas fuentes, pero con evidencia antigua, **aumentan la incertidumbre**, y las notas ganan peso relativo.
- **Sin núcleo**, el sistema no estima.

Todo se puede explicar en una frase al DECE: *"para este estudiante confiamos menos en el test porque es de hace dos años"*.

## 1.8 Contrato del plano cartesiano (se implementa en el Módulo 6)

Para cada área, el modelo entregará el **Interés** como $N(\mu_I, \sigma_I^2)$ y el **Rendimiento** como $N(\mu_R, \sigma_R^2)$, ambos en z. El umbral es $z = 0$ (promedio de la cohorte):

$$P(\text{interés alto}) = \Phi\!\left(\frac{\mu_I}{\sigma_I}\right) \qquad P(\text{rendimiento alto}) = \Phi\!\left(\frac{\mu_R}{\sigma_R}\right)$$

La probabilidad de cada cuadrante es el producto de ambas. Si ningún cuadrante supera el **60 %**, el área se marca **"Indeterminado: falta evidencia"** en lugar de forzar una etiqueta.

| | Rendimiento alto | Rendimiento bajo |
|---|---|---|
| **Interés alto** | 🟢 Vocación consolidada | 🟡 Interés a fortalecer |
| **Interés bajo** | 🔵 Talento no explorado | ⚪ Área no prioritaria |

In [ ]:
# ── 1.8 Función del cuadrante (contrato) ───────────────────────────────
CUADRANTES = {
    (True, True):   '🟢 Vocación consolidada',
    (True, False):  '🟡 Interés a fortalecer',
    (False, True):  '🔵 Talento no explorado',
    (False, False): '⚪ Área no prioritaria',
}
UMBRAL_CERTEZA = 0.60

def cuadrante(mu_i, sd_i, mu_r, sd_r):
    pi, pr = norm.cdf(mu_i / sd_i), norm.cdf(mu_r / sd_r)
    probs = {CUADRANTES[(a, b)]: (pi if a else 1 - pi) * (pr if b else 1 - pr)
             for a in (True, False) for b in (True, False)}
    mejor = max(probs, key=probs.get)
    etiqueta = mejor if probs[mejor] >= UMBRAL_CERTEZA else '❔ Indeterminado: falta evidencia'
    return etiqueta, {k: round(float(v), 2) for k, v in probs.items()}

# Ejemplo: misma posición media, distinta certeza
for caso, args in [('Mucha evidencia', (1.2, 0.3, 1.0, 0.3)), ('Poca evidencia', (1.2, 2.0, 1.0, 2.0))]:
    et, p = cuadrante(*args)
    print(f'{caso:16s} → {et:35s} {p}')

### ✅ Aprobación Módulo 1
- [ ] **CHASIDE** como espacio canónico, con dos dimensiones: Interés y Rendimiento.
- [ ] **Núcleo obligatorio = Notas + CHASIDE.** IPP-J y otras pruebas son complementarias y opcionales.
- [ ] **Escala z** interna y **percentil** para mostrar.
- [ ] **Contrato de fuente**: toda prueba nueva entra con una ficha y una matriz de equivalencia, sin crear estructuras nuevas.
- [ ] **Rúbrica**: criterios y factores. ¿Agregar, quitar o ajustar alguno?
- [ ] **Corrección por familia** (ρ = 0.30 entre autoinformes de intereses).
- [ ] **Umbral del plano**: z = 0 (promedio de la cohorte) y certeza mínima del 60 %.

**Pendientes que no bloquean:** estructura real del IPP-J que usa el DECE · α real de CHASIDE (se calcula en el Módulo 4) · confirmar que el promedio de la cohorte es el umbral correcto, o si conviene usar normas nacionales.

---
# MÓDULO 2 · Estructura de datos en Drive y carga inicial

**Qué resuelve:** un lugar único y ordenado donde se **publican los datos** (entradas), se **configura** el sistema (fichas, mapeos, claves) y se guardan los **resultados**. El notebook nunca edita las entradas; solo las lee.

## 2.1 Estructura de carpetas
```
MyDrive/Colab Notebooks/VocacionalBayesiano_DATOS/     ← junto a este notebook
├── 00_LEEME.md                      ← reglas de uso (se genera aquí)
├── 01_entrada/                      ← AQUÍ SE PUBLICAN LOS DATOS
│   ├── nucleo/
│   │   ├── notas/                   ← transcripts (CSV del sistema académico)        [OBLIGATORIO]
│   │   └── chaside/                 ← respuestas CHASIDE por ítem                    [OBLIGATORIO]
│   └── complementarias/
│       ├── ippj/                    ← IPP-J (si existe)                              [opcional]
│       └── otras/                   ← cualquier otra prueba, una subcarpeta por prueba [opcional]
├── 02_configuracion/                ← reglas editables (sin tocar código)
│   ├── fuentes.json                 ← fichas de cada fuente (Módulo 1)
│   ├── mapeo_materias.csv           ← materia → peso por área CHASIDE
│   └── claves_chaside.csv           ← ítem → área y tipo (interés/aptitud)
├── 03_referencias/
│   └── carreras/carreras_EC.csv     ← carreras por país (reemplazable; se edita abriéndolo con Hojas de cálculo)
├── 04_procesado/                    ← generado por el notebook (no editar)
└── 05_salidas/                      ← resultados para la interfaz y reportes (no editar)
```

## 2.2 Reglas de publicación
1. **Formato:** CSV en UTF-8. Un archivo por entrega.
2. **Nombre:** `fuente_institucion_AAAA-MM.csv`. Ejemplo: `notas_AAI_2026-10.csv`.
3. **Identificador:** cada fila debe tener un ID de estudiante. Si llegan nombres reales, el Módulo 3 los **seudonimiza** antes de cualquier cálculo, y la tabla nombre↔ID nunca sale de `04_procesado/privado/`.
4. **Nunca se sobrescribe:** una entrega nueva es un archivo nuevo, para que el historial (la visión longitudinal) quede trazable.
5. **Una prueba nueva** se agrega con su carpeta en `complementarias/` y su ficha en `fuentes.json`. No hace falta tocar el código.

In [ ]:
# ── 2.1 Montar Drive y crear la estructura (idempotente: no borra nada existente) ──
from google.colab import drive
import os, json, re, glob
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/Colab Notebooks/VocacionalBayesiano_DATOS'   # carpeta del Colab
CARPETAS = [
    '01_entrada/nucleo/notas', '01_entrada/nucleo/chaside',
    '01_entrada/complementarias/ippj', '01_entrada/complementarias/otras',
    '02_configuracion', '03_referencias/carreras',
    '04_procesado/privado', '05_salidas',
]
for c in CARPETAS:
    os.makedirs(f'{BASE}/{c}', exist_ok=True)

LEEME = '\n'.join([
    '# VocacionalBayesiano · Datos',
    '- 01_entrada: aquí se publican los datos. nucleo/ es obligatorio (notas + chaside); complementarias/ es opcional.',
    '- Formato CSV UTF-8. Nombre: fuente_institucion_AAAA-MM.csv. Nunca sobrescribir: cada entrega es un archivo nuevo.',
    '- 02_configuracion: reglas editables (fichas de fuentes, mapeo de materias, claves CHASIDE).',
    '- 03_referencias: carreras por país (reemplazable).',
    '- 04_procesado y 05_salidas: los genera el notebook. No editar a mano.',
    '- Privacidad: los nombres se seudonimizan; la tabla nombre<->ID vive solo en 04_procesado/privado/.',
])
with open(f'{BASE}/00_LEEME.md', 'w') as fh:
    fh.write(LEEME)

for raiz, dirs, _ in sorted(os.walk(BASE)):
    nivel = raiz.replace(BASE, '').count('/')
    print('   ' * nivel + '📁 ' + (os.path.basename(raiz) or 'VocacionalBayesiano_DATOS'))

## 2.2 Archivos de configuración iniciales

Se crean **solo si no existen**, para no pisar ediciones posteriores. Todos son editables por el equipo o el DECE sin tocar código.

**a) `mapeo_materias.csv`: cómo una materia aporta a cada área.**

- Se busca por **palabras clave** (regex) y no por nombre exacto. Así "4th Grade Mathematics", "Algebra II" y "Matemáticas" caen todas en la misma regla. (En el notebook anterior, 100 de 103 materias quedaban sin mapear por buscar el nombre exacto.)
- Las reglas se evalúan **en orden** y gana la primera que coincide. Primero van las exclusiones, luego las reglas específicas y al final las generales.
- **Pesos de 0 a 3** (0 = no aporta, 3 = aporte central). Son **criterio experto provisional**: deben validarse con el DECE y los docentes, y en el Módulo 8 se contrastarán con los datos.
- **Excluidas** (según la propuesta): conducta, formación cristiana, educación física y actividades extracurriculares.

**b) `claves_chaside.csv`: qué ítem pertenece a qué área y si mide interés o aptitud.** CHASIDE tiene 98 ítems: 7 áreas × (10 de interés + 4 de aptitud).

> ⚠️ Estas claves corresponden a la versión de CHASIDE más difundida. **Hay que verificarlas contra la plantilla que usa el DECE** antes de calcular. La celda ya comprueba la estructura (98 ítems únicos, 10 + 4 por área).

**c) `fuentes.json`: las fichas del Módulo 1**, separadas en núcleo y complementarias.

In [ ]:
# ── 2.2 Configuración inicial (solo si no existe) ─────────────────────────
CFG = f'{BASE}/02_configuracion'

def crear_si_no_existe(ruta, df=None, obj=None):
    if os.path.exists(ruta):
        print(f'· ya existe, se conserva: {os.path.basename(ruta)}'); return
    if df is not None:
        df.to_csv(ruta, index=False, encoding='utf-8')
    else:
        with open(ruta, 'w') as fh:
            json.dump(obj, fh, ensure_ascii=False, indent=2)
    print(f'✓ creado: {os.path.basename(ruta)}')

# a) Mapeo materia → área (orden = prioridad)
#   columnas: patron (regex, minúsculas), incluir, C H A S I D E (0-3), justificacion
MAPEO = [
    # --- exclusiones ---
    (r'conduct|conducta',                                   'no', 0,0,0,0,0,0,0, 'No académica'),
    (r'bible|biblia|chapel|christian|cristian',             'no', 0,0,0,0,0,0,0, 'Formación cristiana (excluida por propuesta)'),
    (r'physical ed|\bp\.?e\.?\b|educaci[oó]n f[ií]sica',    'no', 0,0,0,0,0,0,0, 'Educación física (excluida por propuesta)'),
    (r'homeroom|advisory|study hall|extracurric|club',      'no', 0,0,0,0,0,0,0, 'No académica / extracurricular'),
    # --- específicas ---
    (r'computer|programming|robotic|coding|inform[aá]tica|technology|tecnolog', 'si', 0,0,0,0,3,0,1, 'Pensamiento computacional → Ingenierías'),
    (r'econom|business|accounting|contab|finance|emprend',  'si', 3,1,0,0,0,0,1, 'Gestión y números aplicados → Administrativas'),
    (r'psycholog|sociolog|psicolog',                        'si', 0,2,0,2,0,0,0, 'Conducta humana → Humanísticas y Salud'),
    (r'health|salud|anatomy|anatom',                        'si', 0,0,0,3,0,0,1, 'Salud'),
    (r'calculus|c[aá]lculo|statistic|estad[ií]stic',        'si', 1,0,0,0,2,0,3, 'Matemática avanzada → Exactas'),
    (r'math|algebra|geometr|matem|trigonom',                'si', 1,0,0,0,2,0,3, 'Matemática → Exactas e Ingenierías'),
    (r'physic|f[ií]sica',                                   'si', 0,0,0,0,3,0,3, 'Física → Ingenierías y Exactas'),
    (r'chemi|qu[ií]mica',                                   'si', 0,0,0,2,1,0,3, 'Química → Exactas y Salud'),
    (r'biolog|life science|ciencias de la vida',            'si', 0,0,0,3,0,0,2, 'Biología → Salud y Agrarias'),
    # --- generales ---
    (r'science|ciencias',                                   'si', 0,0,0,1,1,0,2, 'Ciencias generales'),
    (r'\bart\b|arts|arte|dibujo|drawing|design|dise[nñ]o|paint', 'si', 0,0,3,0,0,0,0, 'Artes visuales'),
    (r'music|m[uú]sica|band|choir|coro',                    'si', 0,0,3,0,0,0,0, 'Música'),
    (r'drama|theat|teatro',                                 'si', 0,1,3,0,0,0,0, 'Artes escénicas'),
    (r'histor|social stud|estudios sociales|government|civic|geograph|c[ií]vica', 'si', 1,3,0,0,0,1,0, 'Sociedad e instituciones → Humanísticas'),
    (r'english|language arts|reading|writing|literat|ingl[eé]s|lengua|spanish|espa[nñ]ol|french|franc', 'si', 1,3,1,0,0,0,0, 'Lenguaje y comunicación → Humanísticas'),
]
df_mapeo = pd.DataFrame(MAPEO, columns=['patron', 'incluir'] + CODIGOS + ['justificacion'])
crear_si_no_existe(f'{CFG}/mapeo_materias.csv', df=df_mapeo)

# b) Claves CHASIDE (VERIFICAR con la plantilla del DECE)
CLAVES = {
    'C': {'interes': [98,12,64,53,85,1,78,20,71,91], 'aptitud': [15,51,2,46]},
    'H': {'interes': [9,34,80,25,95,67,41,74,56,89], 'aptitud': [63,30,72,86]},
    'A': {'interes': [21,45,96,57,28,11,50,3,81,36], 'aptitud': [22,39,76,82]},
    'S': {'interes': [33,92,70,8,87,62,23,44,16,52], 'aptitud': [69,40,29,4]},
    'I': {'interes': [75,6,19,38,60,27,83,54,47,97], 'aptitud': [26,59,90,10]},
    'D': {'interes': [84,31,48,73,5,65,14,37,58,24], 'aptitud': [13,66,18,43]},
    'E': {'interes': [77,42,88,17,93,32,68,49,35,61], 'aptitud': [94,7,79,55]},
}
df_claves = pd.DataFrame([(it, a, t) for a, d in CLAVES.items() for t, its in d.items() for it in its],
                         columns=['item', 'area', 'tipo']).sort_values('item')
conteo = df_claves.groupby(['area', 'tipo']).size().unstack()
chk = {'98 ítems': len(df_claves) == 98,
       'sin repetidos': df_claves['item'].nunique() == 98,
       'numerados 1-98': sorted(df_claves['item']) == list(range(1, 99)),
       '10 interés + 4 aptitud por área': bool((conteo['interes'] == 10).all() and (conteo['aptitud'] == 4).all())}
print('Estructura CHASIDE:', ' · '.join(f"{k} {'✓' if v else '✗'}" for k, v in chk.items()))
crear_si_no_existe(f'{CFG}/claves_chaside.csv', df=df_claves)

# c) Fichas de fuentes (núcleo + complementarias). En el Módulo 5 se leerán desde aquí.
FICHAS = {
    'nucleo': [
        {'id': 'notas', 'tipo': 'rendimiento', 'alfa': 0.85, 'familia': 'desempeno_academico',
         'criterios': {'validacion': 'publicada', 'equivalencia': 'debil'}, 'carpeta': '01_entrada/nucleo/notas'},
        {'id': 'chaside_int', 'tipo': 'interes', 'alfa': 0.80, 'familia': 'autoinforme_intereses',
         'criterios': {'validacion': 'publicada', 'equivalencia': 'directa'}, 'carpeta': '01_entrada/nucleo/chaside'},
        {'id': 'chaside_apt', 'tipo': 'rendimiento', 'alfa': 0.60, 'familia': 'autoinforme_aptitud',
         'criterios': {'validacion': 'publicada', 'equivalencia': 'directa'}, 'carpeta': '01_entrada/nucleo/chaside'},
    ],
    'complementarias': [
        {'id': 'ippj', 'tipo': 'interes', 'alfa': 0.85, 'familia': 'autoinforme_intereses',
         'criterios': {'validacion': 'publicada', 'equivalencia': 'aproximada'},
         'carpeta': '01_entrada/complementarias/ippj', 'equivalencia': 'PENDIENTE: estructura real del IPP-J'},
    ],
    '_nota': 'Valores de alfa PROVISIONALES. Agregar una prueba = agregar una ficha en complementarias.',
}
crear_si_no_existe(f'{CFG}/fuentes.json', obj=FICHAS)

## 2.3 Carga inicial: referencia de carreras

Publica la tabla de 84 carreras con sus pesos CHASIDE en `03_referencias/carreras/carreras_EC.csv`. Es la **única versión**: para editarla, se abre ese CSV con Hojas de cálculo de Google. Si el archivo ya existe, no se vuelve a copiar.

Las notas y el CHASIDE se publican con la celda 2.7 (entregas en Excel o CSV) o directamente en `01_entrada/nucleo/`.

In [ ]:
# ── 2.3 Carga inicial: referencia de carreras ───────────────────────────────
ORIGENES = [
    ('https://docs.google.com/spreadsheets/d/1K0otKBBv4a8HZAlgcjGTYMT-lO05J8FkzeGfpDEUzo4/export?format=csv&gid=0',
     '03_referencias/carreras/carreras_EC.csv'),
]
for url, destino in ORIGENES:
    ruta = f'{BASE}/{destino}'
    if os.path.exists(ruta):
        print(f'· ya existe: {destino}'); continue
    try:
        pd.read_csv(url).to_csv(ruta, index=False, encoding='utf-8')
        print(f'✓ publicado: {destino}')
    except Exception as e:
        print(f'✗ no se pudo copiar {destino}: {e}')

## 2.4 Inventario y diagnóstico de lo publicado

Es una función reutilizable: se corre cada vez que se publica algo nuevo. Para cada archivo revisa:

- filas y columnas,
- si tiene columna de ID,
- si hay **nombres reales** (alerta de privacidad),
- cuántos ítems se detectan en las pruebas.

Así se sabe **qué hay que resolver antes de configurar el algoritmo**.

In [ ]:
# ── 2.4 Inventario ────────────────────────────────────────────────────────
def inventario():
    filas = []
    for raiz, _, archivos in os.walk(f'{BASE}/01_entrada'):
        if '/_' in raiz: continue   # carpetas archivadas (_archivo_...) no cuentan
        for a in sorted(archivos):
            if not a.endswith('.csv'): continue
            ruta = os.path.join(raiz, a)
            df = pd.read_csv(ruta)
            rel = ruta.replace(BASE + '/01_entrada/', '')
            col_id = next((c for c in df.columns if re.search(r'id|nombre|student|estudiante', str(c), re.I)), None)
            nombres = col_id is not None and df[col_id].astype(str).str.contains(r'[A-Za-zÁÉÍÓÚáéíóúñ]+ [A-Za-zÁÉÍÓÚáéíóúñ]+').any()
            items = [c for c in df.columns if re.search(r'\d', str(c)) and re.fullmatch(r'\D{0,6}\d{1,2}', str(c).strip())]
            filas.append({
                'archivo': rel, 'filas': len(df), 'columnas': df.shape[1],
                'col_id': col_id, 'estudiantes': df[col_id].nunique() if col_id else None,
                'nombres_reales': '⚠️ sí' if nombres else 'no',
                'items_detectados': len(items) if ('chaside' in rel or 'ippj' in rel) else '—',
            })
    return pd.DataFrame(filas)

inv = inventario()
display(inv)

print('\nNÚCLEO (obligatorio)')
for fuente in ['nucleo/notas', 'nucleo/chaside']:
    ok = len(inv) > 0 and inv['archivo'].str.startswith(fuente).any()
    print(f"  {'✓' if ok else '✗ FALTA'} {fuente}")
print('COMPLEMENTARIAS (opcionales)')
for fuente in ['complementarias/ippj', 'complementarias/otras']:
    n = int(inv['archivo'].str.startswith(fuente).sum()) if len(inv) else 0
    print(f"  {'✓' if n else '·'} {fuente}: {n} archivo(s)")

In [ ]:
# ── 2.5 Vista rápida de lo publicado (para verificar formato) ──
for ruta in sorted(glob.glob(f'{BASE}/01_entrada/nucleo/*/*.csv')) + [f'{BASE}/03_referencias/carreras/carreras_EC.csv']:
    if not os.path.exists(ruta): continue
    df = pd.read_csv(ruta, nrows=5)
    print(f"\n▶ {ruta.replace(BASE + '/', '')}: {df.shape[1]} columnas")
    print('  Primeras columnas:', list(df.columns[:12]))
    print('  Últimas columnas :', list(df.columns[-10:]))

## 2.6 Control de calidad del CHASIDE publicado

Si un archivo CHASIDE trae, además de las 98 respuestas, totales por área ya calculados (`C_Administrativas`, …), esta celda los recalcula desde las respuestas con `claves_chaside.csv` y los compara.

- **Si coinciden:** las claves corresponden a la versión del DECE.
- **Si no coinciden:** las claves no corresponden, o los totales no salen de las respuestas.

**Regla:** el sistema siempre calcula desde las respuestas y nunca confía en totales precalculados.

In [ ]:
# ── 2.6 Totales recalculados vs. totales del archivo ──
COL_TOTAL = {'C': 'C_Administrativas', 'H': 'H_Humanidades', 'A': 'A_Artes', 'S': 'S_Salud',
             'I': 'I_Ingenieria', 'D': 'D_Defensa', 'E': 'E_Exactas'}
claves_chk = pd.read_csv(f'{CFG}/claves_chaside.csv')
for ruta in sorted(glob.glob(f'{BASE}/01_entrada/nucleo/chaside/*.csv')):
    ch_chk = pd.read_csv(ruta, dtype=str, encoding='utf-8-sig')
    if not set(COL_TOTAL.values()) & set(ch_chk.columns):
        print(f'{os.path.basename(ruta)}: sin totales precalculados → se calculan desde las 98 respuestas ✓'); continue
    si_chk = ch_chk[[f'P{i}' for i in range(1, 99)]].apply(lambda s: s.str.strip().str.lower().isin(['sí', 'si', '1', 'true']))
    ok = tot = 0
    for a in CODIGOS:
        its = [f'P{i}' for i in claves_chk.query('area == @a')['item']]
        ok += (si_chk[its].sum(axis=1) == pd.to_numeric(ch_chk[COL_TOTAL[a]])).sum(); tot += len(ch_chk)
    print(f'{os.path.basename(ruta)}: totales coincidentes {ok}/{tot}' + ('' if ok == tot else ' ⚠️ no salen de las respuestas → se usan los recalculados'))

## 2.7 Publicar una entrega recibida como archivo (Excel o CSV)

Cuando una entrega llega como archivo suelto (por ejemplo, el export de la API del sistema académico en `.xlsx`), esta celda:

1. la convierte a **CSV UTF-8**,
2. le pone el **nombre normalizado**,
3. la publica en la carpeta correcta **sin sobrescribir** nada.

Entrega publicada aquí: **`datos_api_simulados_500.xlsx`**, 500 estudiantes simulados con IDs numéricos y el mismo formato del transcript.

In [ ]:
# ── 2.7 Publicar entregas sueltas ─────────────────────────────────────────
ENTREGAS = [  # (archivo recibido en /content, carpeta destino, nombre normalizado)
    ('/content/datos_api_simulados_500.xlsx', '01_entrada/nucleo/notas', 'notas_AAI-sim500_2026-10.csv'),
]
for origen, carpeta, nombre in ENTREGAS:
    destino = f'{BASE}/{carpeta}/{nombre}'
    if os.path.exists(destino):
        print(f'· ya publicado: {carpeta}/{nombre}'); continue
    if not os.path.exists(origen):
        print(f'✗ no está en /content: {os.path.basename(origen)} (súbelo con el panel de archivos)'); continue
    df = pd.read_excel(origen) if origen.endswith(('.xlsx', '.xls')) else pd.read_csv(origen)
    df.to_csv(destino, index=False, encoding='utf-8')
    print(f'✓ publicado: {carpeta}/{nombre} · {len(df)} filas · {df.iloc[:, 0].nunique()} estudiantes')

display(inventario())

### ✅ Aprobación Módulo 2
- [ ] La estructura de carpetas sirve para publicar datos (núcleo obligatorio y complementarias opcionales).
- [ ] Reglas de publicación: CSV UTF-8, nombre `fuente_institucion_AAAA-MM`, nunca sobrescribir.
- [ ] `mapeo_materias.csv`: los pesos 0–3 y las exclusiones son razonables (validar con el DECE).
- [ ] `claves_chaside.csv`: verificar contra la plantilla oficial del DECE.
- [ ] Revisar el inventario: ¿qué falta o qué hay que corregir antes de configurar el algoritmo?

---
# MÓDULO 3 · Limpieza de calificaciones → rendimiento por área

**Qué resuelve:** convierte el transcript crudo (todas las materias, todos los grados, escalas mezcladas, nombres reales) en una **evidencia de rendimiento por área CHASIDE**, en escala z, con trazabilidad de qué materias aportaron a cada área.

**Pipeline (cada paso es una celda):**

| Paso | Qué hace | Criterio |
|---|---|---|
| 3.1 | Carga todas las entregas de `nucleo/notas/` | Une las entregas y elimina filas duplicadas |
| 3.2 | **Seudonimiza** | Nombre → ID aleatorio. La llave queda solo en `04_procesado/privado/` |
| 3.3 | Filtra **secundaria (9.º a 12.º)** | Alcance de la propuesta |
| 3.4 | **Clasifica materias** con `mapeo_materias.csv` | Regex en orden: primero exclusiones, luego reglas. Reporta las materias sin regla |
| 3.5 | **Normaliza la escala** a 0–10 | Detecta la escala por fila. Un 0 con letra A/B/C es "sin calificación", no un cero |
| 3.6 | Une semestres | Promedio por estudiante × grado × materia |
| 3.7 | **Estandariza (z)** contra la cohorte | Por materia y grado si hay ≥ 5 estudiantes; si no, contra todo el grado |
| 3.8 | **Agrega por área y por año** | Promedio ponderado por los pesos materia→área |
| 3.9 | **Combina los años** | Los años recientes pesan más (γ = 0.8) |
| 3.10 | Guarda y diagnostica | Resultados en `04_procesado/` y control de cobertura por área |

In [ ]:
# ── 3.0 Mapeo v2: reglas detectadas al revisar las materias reales ────────────
# Al revisar el transcript aparecen materias que la v1 no cubría. Se AGREGAN (sin borrar nada) y se registra el cambio.
mapeo = pd.read_csv(f'{CFG}/mapeo_materias.csv')
NUEVAS = [
    # exclusiones (extracurriculares / formación) → van ANTES de todo
    (r'student council|stuco|class officer|model united|\bmun\b|basketball|volleyball|football|soccer|varsity|\bjv\b|discipleship',
     'no', 0,0,0,0,0,0,0, 'v2 · Extracurricular / gobierno estudiantil / deporte / formación'),
    # lenguaje: composición y escritura
    (r'composition|narrativ|escritura|redacci',
     'si', 1,3,1,0,0,0,0, 'v2 · Composición y escritura → Humanísticas'),
]
agregadas = []
for regla in NUEVAS:
    if regla[0] not in mapeo['patron'].values:
        fila = pd.DataFrame([regla], columns=mapeo.columns)
        mapeo = pd.concat([fila, mapeo], ignore_index=True) if regla[1] == 'no' else \
                pd.concat([mapeo.iloc[:4], fila, mapeo.iloc[4:]], ignore_index=True)
        agregadas.append(regla[-1])
if agregadas:
    mapeo.to_csv(f'{CFG}/mapeo_materias.csv', index=False, encoding='utf-8')
print('Reglas agregadas:', agregadas or 'ninguna (ya estaban)')
print(f'Mapeo actual: {len(mapeo)} reglas')

In [ ]:
# ── 3.1 Carga de todas las entregas de notas ──────────────────────────────
import glob, secrets
archivos = sorted(glob.glob(f'{BASE}/01_entrada/nucleo/notas/*.csv'))
crudo = pd.concat([pd.read_csv(a).assign(_archivo=os.path.basename(a)) for a in archivos], ignore_index=True)
antes = len(crudo)
crudo = crudo.drop_duplicates(subset=[c for c in crudo.columns if c != '_archivo'])
print(f'{len(archivos)} entrega(s) · {antes} filas · {antes - len(crudo)} duplicadas eliminadas · {len(crudo)} filas')

COL = {'est': 'Student ID (System)', 'grado': 'Grade Level 1', 'anio': 'Year',
       'materia': 'Course', 'nota': 'Final grade AVG', 'letra': 'Final grade'}

### 3.2 Seudonimización

Cada estudiante recibe un ID aleatorio (`E-xxxxxx`). La tabla nombre↔ID se guarda **solo** en `04_procesado/privado/llave_ids.csv` y se reutiliza: el mismo estudiante conserva su ID en entregas futuras, lo que permite el seguimiento longitudinal.

Desde aquí **ningún cálculo ni salida usa nombres** (Ley Orgánica de Protección de Datos Personales y protección de menores).

In [ ]:
# ── 3.2 Seudonimización ────────────────────────────────────────────────────
RUTA_LLAVE = f'{BASE}/04_procesado/privado/llave_ids.csv'
llave = pd.read_csv(RUTA_LLAVE) if os.path.exists(RUTA_LLAVE) else pd.DataFrame(columns=['nombre', 'id'])
nuevos = [n for n in crudo[COL['est']].astype(str).unique() if n not in set(llave['nombre'])]
usados = set(llave['id'])
for n in nuevos:
    nid = f'E-{secrets.token_hex(3)}'
    while nid in usados: nid = f'E-{secrets.token_hex(3)}'
    usados.add(nid); llave.loc[len(llave)] = [n, nid]
llave.to_csv(RUTA_LLAVE, index=False)

notas = crudo.assign(id=crudo[COL['est']].astype(str).map(dict(zip(llave['nombre'], llave['id'])))) \
             .drop(columns=[COL['est']])
print(f'{len(nuevos)} ID nuevo(s) · {len(llave)} en la llave · nombres eliminados del dataset de trabajo ✓')

In [ ]:
# ── 3.3 Filtro de nivel: secundaria 9.º–12.º ────────────────────────────────
notas['grado'] = pd.to_numeric(notas[COL['grado']], errors='coerce')
fuera = notas[~notas['grado'].between(9, 12)]
notas = notas[notas['grado'].between(9, 12)].copy()
print(f'Se conservan {len(notas)} filas de secundaria · se excluyen {len(fuera)} de otros niveles '
      f'({", ".join(sorted(fuera[COL["grado"]].astype(str).unique()))})')
print('Grados presentes por estudiante:')
display(notas.groupby('id')['grado'].agg(lambda g: sorted(g.unique().astype(int))).rename('grados'))
faltan_12 = notas.groupby('id')['grado'].max().lt(12).sum()
if faltan_12: print(f'ℹ️ {faltan_12} estudiante(s) sin notas de 12.º (semestre en curso) → se usa lo disponible.')

### 3.4 Clasificación de materias

Cada materia se compara con las reglas de `mapeo_materias.csv` **en orden**, y gana la primera que coincide.

- **Excluida:** la regla dice `incluir = no` (conducta, formación cristiana, educación física, extracurriculares).
- **Sin regla:** ninguna regla coincide. **No se inventa un peso**: se reporta para agregar una regla. Mientras tanto no aporta (ausencia ≠ cero).

In [ ]:
# ── 3.4 Clasificación de materias ──────────────────────────────────────────
def clasificar(materia):
    m = str(materia).lower().strip()
    for i, r in mapeo.iterrows():
        if re.search(r['patron'], m):
            return i, r['incluir'], r['justificacion']
    return None, 'sin_regla', '—'

cat = pd.DataFrame([(mat, *clasificar(mat)) for mat in sorted(notas[COL['materia']].astype(str).unique())],
                   columns=['materia', 'regla', 'estado', 'justificacion'])
cat['estado'] = cat['estado'].replace({'si': 'incluida', 'no': 'excluida'})
for estado in ['incluida', 'excluida', 'sin_regla']:
    sub = cat[cat['estado'] == estado]
    print(f'\n{estado.upper()} ({len(sub)})')
    for _, r in sub.iterrows():
        print(f"   {r['materia'][:55]:55s} → {r['justificacion']}")
notas = notas.merge(cat[['materia', 'regla', 'estado']], left_on=COL['materia'], right_on='materia', how='left')

### 3.5 Normalización de escala

El transcript mezcla escalas (0–100 en preescolar, 0–4 en primaria y 0–10 en secundaria). Cada fila se lleva a **0–10** según su valor:

| Valor | Escala detectada | Conversión |
|---|---|---|
| > 10 | 0–100 | ÷ 10 |
| 4 < valor ≤ 10 | 0–10 | igual |
| ≤ 4 con GPA informado | 0–4 (GPA) | × 2.5 |
| **0 con letra A, B o C** | **sin calificación** (actividad no evaluada) | **NaN**, no cero |

Es la misma lógica que en el Módulo 1: un dato ausente no es un dato bajo.

In [ ]:
# ── 3.5 Normalización de escala ────────────────────────────────────────────
def a_0_10(fila):
    v, letra = fila[COL['nota']], str(fila[COL['letra']]).strip().upper()
    if pd.isna(v): return np.nan, 'vacía'
    if v == 0 and letra[:1] in 'ABC': return np.nan, 'sin calificación (0 con letra aprobatoria)'
    if v > 10: return v / 10, '0-100'
    if v > 4:  return v, '0-10'
    return v * 2.5, '0-4'

notas[['nota10', 'escala']] = notas.apply(lambda f: pd.Series(a_0_10(f)), axis=1)
display(notas['escala'].value_counts().rename('filas por escala detectada'))

In [ ]:
# ── 3.6 Unir semestres: promedio por estudiante × grado × materia ──────────
validas = notas[(notas['estado'] == 'incluida') & notas['nota10'].notna()]
por_materia = validas.groupby(['id', 'grado', COL['materia'], 'regla'], as_index=False)['nota10'].mean()
print(f'{len(validas)} filas válidas → {len(por_materia)} registros estudiante×grado×materia')

### 3.7 Estandarización contra la cohorte

$$z_{e,s,t} = \frac{nota_{e,s,t} - \bar{x}_{s,t}}{s_{s,t}}$$

donde $s$ es la materia y $t$ el grado.

- **Por qué por materia:** un 9 en Física (difícil) vale más que un 9 en una materia donde todos sacan 9.5.
- **Respaldo:** con menos de **5 estudiantes** por materia y grado, la desviación no es estable. En ese caso se estandariza contra **todas las notas del grado**. Con los 3 estudiantes actuales siempre se usa el respaldo; con los ~200 reales se usará la regla por materia.

In [ ]:
# ── 3.7 Estandarización z ──────────────────────────────────────────────────
N_MIN = 5
g_mat = por_materia.groupby(['grado', COL['materia']])['nota10']
g_gra = por_materia.groupby('grado')['nota10']
por_materia['n_cohorte'] = g_mat.transform('count')
usa_materia = por_materia['n_cohorte'] >= N_MIN
media = np.where(usa_materia, g_mat.transform('mean'), g_gra.transform('mean'))
sd    = np.where(usa_materia, g_mat.transform('std'),  g_gra.transform('std'))
por_materia['z'] = (por_materia['nota10'] - media) / np.where(sd > 0, sd, np.nan)
por_materia['referencia_z'] = np.where(usa_materia, 'materia×grado', 'grado (respaldo, n<5)')
display(por_materia['referencia_z'].value_counts())

### 3.8 Rendimiento por área y por año

Para cada estudiante $e$, área $a$ y grado $t$:

$$R_{e,a,t} = \frac{\sum_s w_{s,a}\, z_{e,s,t}}{\sum_s w_{s,a}}$$

donde $w_{s,a}$ es el peso (0–3) de la materia $s$ hacia el área $a$, tomado de `mapeo_materias.csv`.

- Si ninguna materia aporta al área, $R$ = **NaN**: área sin evidencia académica.
- Además se guarda $W_{e,a} = \sum w$, la **cantidad de evidencia**. El Módulo 5 la usará para ajustar la confianza: un área sustentada por 8 materias es más confiable que una sustentada por 1.

In [ ]:
# ── 3.8 Agregación materia → área, por año ─────────────────────────────────
W = mapeo[CODIGOS].astype(float)
filas = []
for _, r in por_materia.dropna(subset=['z']).iterrows():
    for a in CODIGOS:
        w = W.loc[r['regla'], a]
        if w > 0:
            filas.append({'id': r['id'], 'grado': int(r['grado']), 'area': a, 'materia': r[COL['materia']],
                          'w': w, 'z': r['z'], 'wz': w * r['z']})
aporte = pd.DataFrame(filas)
por_anio = aporte.groupby(['id', 'area', 'grado']).agg(wz=('wz', 'sum'), W=('w', 'sum'), n_materias=('materia', 'nunique')).reset_index()
por_anio['R'] = por_anio['wz'] / por_anio['W']
display(por_anio.pivot_table(index=['id', 'grado'], columns='area', values='R').reindex(columns=CODIGOS).round(2))

### 3.9 Combinación temporal

$$R_{e,a} = \frac{\sum_t \lambda_t\, W_{e,a,t}\, R_{e,a,t}}{\sum_t \lambda_t\, W_{e,a,t}} \qquad \lambda_t = \gamma^{\,(T_e - t)}, \quad \gamma = 0.8$$

- $T_e$ es el último grado con notas del estudiante. El año más reciente pesa 1, el anterior 0.8 y el siguiente 0.64.
- **Por qué:** el rendimiento reciente refleja mejor la situación actual, pero la trayectoria no se descarta. Esto materializa la **visión longitudinal** de la propuesta.
- γ es un parámetro declarado. En el Módulo 8 se comparará γ = 1 (todos los años iguales) contra γ = 0.8.

Se reportan dos lecturas:

- **Absoluta** ($R$): comparada con la cohorte. Es la que alimenta el **plano cartesiano**.
- **Relativa** ($R - \bar{R}_e$): las fortalezas propias del estudiante comparadas con sus otras áreas. Sirve para orientar aunque el estudiante sea alto o bajo en todo.

In [ ]:
# ── 3.9 Combinación temporal ───────────────────────────────────────────────
GAMMA = 0.8
por_anio['T'] = por_anio.groupby('id')['grado'].transform('max')
por_anio['lambda'] = GAMMA ** (por_anio['T'] - por_anio['grado'])
por_anio['peso'] = por_anio['lambda'] * por_anio['W']
rend = por_anio.groupby(['id', 'area']).apply(
    lambda g: pd.Series({'R': (g['peso'] * g['R']).sum() / g['peso'].sum(),
                         'W_total': g['W'].sum(), 'W_efectiva': g['peso'].sum(),
                         'anios': len(g), 'n_materias': g['n_materias'].sum()}), include_groups=False).reset_index()
# Todas las áreas para todos: las que no tienen evidencia quedan NaN (no cero)
rend = pd.MultiIndex.from_product([rend['id'].unique(), CODIGOS], names=['id', 'area']).to_frame(index=False) \
         .merge(rend, how='left')
rend['R_relativo'] = rend['R'] - rend.groupby('id')['R'].transform('mean')
rend['percentil'] = z_a_percentil(rend['R']).round(0)

print('RENDIMIENTO ABSOLUTO (z vs cohorte) — alimenta el plano cartesiano')
display(rend.pivot(index='id', columns='area', values='R').reindex(columns=CODIGOS).round(2).head(10))
print('RENDIMIENTO RELATIVO (fortalezas propias del estudiante)')
display(rend.pivot(index='id', columns='area', values='R_relativo').reindex(columns=CODIGOS).round(2).head(10))
print('EVIDENCIA (suma de pesos efectiva) — base de la confianza en el Módulo 5')
display(rend.pivot(index='id', columns='area', values='W_efectiva').reindex(columns=CODIGOS).round(1).head(10))
print(f'(se muestran 10 de {rend.id.nunique()} estudiantes)')

In [ ]:
# ── 3.10 Explicación por estudiante: qué materias sostienen cada área ──────
def explicar_rendimiento(id_est, top=3):
    sub = aporte[aporte['id'] == id_est]
    print(f'▶ {id_est}')
    for a in CODIGOS:
        r = rend[(rend['id'] == id_est) & (rend['area'] == a)].iloc[0]
        if pd.isna(r['R']):
            print(f'   {a} {EJES[a][:28]:28s}  sin evidencia académica'); continue
        s = sub[sub['area'] == a].groupby('materia').agg(w=('w', 'first'), z=('z', 'mean')).sort_values('w', ascending=False)
        detalle = ', '.join(f'{m[:22]} (w={w:.0f}, z={z:+.2f})' for m, (w, z) in s.head(top).iterrows())
        print(f"   {a} {EJES[a][:28]:28s}  R={r['R']:+.2f} (P{r['percentil']:.0f}) · {int(r['n_materias'])} materias · {detalle}")

for e in rend['id'].unique()[:3]:          # ejemplo con 3 estudiantes; explicar_rendimiento('E-xxxxxx') para cualquiera
    explicar_rendimiento(e)

In [ ]:
# ── 3.11 Guardar y diagnosticar ────────────────────────────────────────────
PROC = f'{BASE}/04_procesado'
rend.to_csv(f'{PROC}/rendimiento_por_area.csv', index=False)
por_anio.to_csv(f'{PROC}/rendimiento_por_area_anio.csv', index=False)
aporte.to_csv(f'{PROC}/trazabilidad_materia_area.csv', index=False)
cat.to_csv(f'{PROC}/clasificacion_materias.csv', index=False)
print('✓ Guardado en 04_procesado/: rendimiento_por_area · rendimiento_por_area_anio · trazabilidad_materia_area · clasificacion_materias')

print('\nDIAGNÓSTICO')
cob = rend.groupby('area')['R'].apply(lambda s: s.notna().mean() * 100).reindex(CODIGOS)
for a, p in cob.items():
    print(f"   {a} {EJES[a][:35]:35s} cobertura {p:5.0f}%  {'⚠️ sin materias que lo sustenten' if p == 0 else ''}")
print(f"   Materias sin regla: {(cat['estado'] == 'sin_regla').sum()}")

### ✅ Aprobación Módulo 3
- [ ] Seudonimización: llave aparte y nombres fuera del dataset de trabajo.
- [ ] Solo secundaria (9.º–12.º). Un 0 con letra aprobatoria se trata como "sin calificación".
- [ ] Mapeo v2: ¿están bien las exclusiones nuevas (gobierno estudiantil, deportes, MUN, discipulado)?
- [ ] Estandarización z: por materia×grado con n ≥ 5; si no, contra todo el grado.
- [ ] Agregación ponderada por los pesos materia→área y combinación temporal con γ = 0.8.
- [ ] Dos lecturas: **absoluta** (para el plano) y **relativa** (fortalezas propias).
- [ ] ⚠️ **Evidencia desigual por área.** **D (Defensa)** solo se sostiene en Historia con peso 1 (evidencia ≈ 2.4, frente a ≈ 24 de H). **S (Salud)** solo tiene 3 materias y ninguna en 11.º. El Módulo 5 les dará **menos confianza** automáticamente. ¿Es aceptable, o hay materias del currículo de AAI que deban aportar a D o a S?

---
# MÓDULO 3B · Simulación CHASIDE y escenarios de estrés (solo para probar el modelo)

> ⚠️ **Solo para pruebas.** Los 500 estudiantes todavía no tienen CHASIDE real. Este módulo genera respuestas **plausibles** para poder probar el modelo.
> - Se publica **un solo archivo** simulado (escenario **realista**) en `01_entrada/nucleo/chaside/`. Cuando lleguen las pruebas reales, se reemplaza por el CHASIDE real de cada estudiante y nada más cambia.
> - Los otros escenarios (**alineado, desalineado, estrés**) **no se guardan**: se generan en memoria solo en la sección 5.F, para comprobar que el modelo resiste datos difíciles.

## Modelo generativo (decisión B: el rendimiento se ancla en las notas)

Para cada estudiante $e$ y área $a$ se generan tres rasgos "verdaderos":

**1 · Rendimiento verdadero**, coherente con sus notas reales (con la confiabilidad que las notas muestran en los datos):

$$R^*_{e,a} = \sqrt{\alpha_{notas}}\; z^{notas}_{e,a} + \sqrt{1-\alpha_{notas}}\;\varepsilon$$

**2 · Autopercepción de aptitud**, lo que el estudiante *cree* de sí mismo. Se parece a su rendimiento en grado $\lambda$, más un **sesgo personal** $b_e$ (hay quien se sobreestima en todo y quien se subestima):

$$A^*_{e,a} = \lambda\, R^*_{e,a} + \sqrt{1-\lambda^2}\;\eta + b_e \qquad b_e \sim N(0, \sigma_{sesgo})$$

**3 · Interés verdadero**, relacionado con el rendimiento en grado $\rho$:

$$I^*_{e,a} = \rho\, R^*_{e,a} + \sqrt{1-\rho^2}\;\nu$$

**4 · Respuestas** (modelo logístico, como en Teoría de Respuesta al Ítem). Los ítems de interés responden a $I^*$ y los de aptitud a $A^*$:

$$P(\text{Sí}) = \frac{1}{1+e^{-(b_j + \kappa\,\theta)}}$$

## Escenarios

| Escenario | ρ (interés ~ rendimiento) | λ (autopercepción fiel) | Sesgo personal | κ (claridad de ítems) | Respuestas al azar | Uso |
|---|---|---|---|---|---|---|
| **Alineado** | 0.4 | 0.6 | 0 | 1.7 | 3 % | Estrés (5.F) |
| **Realista** | 0.2 | 0.4 | 0.3 | 1.7 | 3 % | **Archivo publicado** |
| **Desalineado** | 0.0 | 0.3 | 0.3 | 1.5 | 5 % | Estrés (5.F) |
| **Estrés** | 0.0 | 0.2 | 0.5 | 1.0 | 10 % | Estrés (5.F) |

**Justificación del escenario realista:**

- La relación interés–habilidad en la literatura es baja a moderada (≈ 0.2–0.4; Ackerman y Heggestad, 1997).
- La autoevaluación de capacidades en adolescentes es solo moderadamente precisa.

La verdad ($R^*$, $I^*$, $A^*$) se guarda en `04_procesado/privado/` y **el modelo nunca la ve**. Solo se usa para evaluar.

In [ ]:
# ── 3B.1 Escenarios (parámetros declarados) ──────────────────────────────────
ESCENARIOS = {
    'alineado':    dict(rho_IR=0.4, lam_auto=0.6, sesgo_sd=0.0, kappa=1.7, p_aleatorio=0.03),
    'realista':    dict(rho_IR=0.2, lam_auto=0.4, sesgo_sd=0.3, kappa=1.7, p_aleatorio=0.03),
    'desalineado': dict(rho_IR=0.0, lam_auto=0.3, sesgo_sd=0.3, kappa=1.5, p_aleatorio=0.05),
    'estres':      dict(rho_IR=0.0, lam_auto=0.2, sesgo_sd=0.5, kappa=1.0, p_aleatorio=0.10),
}
ESCENARIO_PUBLICADO = 'realista'
COMUN = dict(b_media=-0.4, b_sd=0.4,
             p_antiguedad={'menos_1_anio': .85, '1_a_2_anios': .10, 'mas_2_anios': .05},
             p_modalidad={'supervisada': .70, 'autonoma': .30},
             p_completitud={'completa': .90, 'parcial': .07, 'incompleta': .03},
             blancos={'completa': 0.0, 'parcial': 0.10, 'incompleta': 0.25})
claves = pd.read_csv(f'{CFG}/claves_chaside.csv').sort_values('item').reset_index(drop=True)
b_item = np.random.default_rng(99).normal(COMUN['b_media'], COMUN['b_sd'], size=len(claves))  # popularidad fija de cada ítem

# Confiabilidad observada de las notas (consistencia entre años, Spearman-Brown) → la "naturaleza" simulada la respeta
def confiabilidad_notas(por_anio, ids=None):
    d = por_anio if ids is None else por_anio[por_anio['id'].isin(ids)]
    out = {}
    for a in CODIGOS:
        w = d[d['area'] == a].pivot_table(index='id', columns='grado', values='R')
        c = w.corr().values[np.triu_indices(w.shape[1], 1)]
        r = np.nanmean(c); k = w.shape[1]
        out[a] = float(np.clip(k * r / (1 + (k - 1) * r), 0.30, 0.95))
    return out

ALFA_NOTAS_OBS = confiabilidad_notas(por_anio)
print('Confiabilidad de las notas por área (consistencia entre años):', {a: round(v, 2) for a, v in ALFA_NOTAS_OBS.items()})

In [ ]:
# ── 3B.2 Generador ───────────────────────────────────────────────────────────
Zn = rend.pivot(index='id', columns='area', values='R').reindex(columns=CODIGOS)
Zn = ((Zn - Zn.mean()) / Zn.std()).fillna(0)                       # notas estandarizadas por área

def simular(nombre, semilla=2026):
    P = ESCENARIOS[nombre]
    rng = np.random.default_rng(semilla)
    n = len(Zn)
    al = np.array([ALFA_NOTAS_OBS[a] for a in CODIGOS])
    R_v = np.sqrt(al) * Zn.values + np.sqrt(1 - al) * rng.standard_normal((n, 7))
    sesgo = rng.normal(0, P['sesgo_sd'], size=(n, 1)) if P['sesgo_sd'] > 0 else np.zeros((n, 1))
    A_v = P['lam_auto'] * R_v + np.sqrt(1 - P['lam_auto']**2) * rng.standard_normal((n, 7)) + sesgo
    I_v = P['rho_IR'] * R_v + np.sqrt(1 - P['rho_IR']**2) * rng.standard_normal((n, 7))

    def elegir(p): return rng.choice(list(p), size=n, p=list(p.values()))
    cond = pd.DataFrame({'id': Zn.index, 'antiguedad': elegir(COMUN['p_antiguedad']),
                         'modalidad': elegir(COMUN['p_modalidad']),
                         'completitud_sim': elegir(COMUN['p_completitud']),
                         'aleatorio': rng.random(n) < P['p_aleatorio']})
    rango = {'menos_1_anio': (1, 11), '1_a_2_anios': (12, 23), 'mas_2_anios': (24, 36)}
    cond['fecha_aplicacion'] = [(pd.Timestamp('2026-10-01') - pd.DateOffset(months=int(rng.integers(*rango[x])))).strftime('%Y-%m-%d')
                                for x in cond['antiguedad']]
    resp = np.empty((n, len(claves)), dtype=object)
    idx = {a: i for i, a in enumerate(CODIGOS)}
    for j, it in claves.iterrows():
        theta = (I_v if it['tipo'] == 'interes' else A_v)[:, idx[it['area']]]
        p = 1 / (1 + np.exp(-(b_item[j] + P['kappa'] * theta)))
        p = np.where(cond['aleatorio'], 0.5, p)
        resp[:, j] = np.where(rng.random(n) < p, 'Sí', 'No')
    for k, c in enumerate(cond['completitud_sim']):
        resp[k, rng.random(len(claves)) < COMUN['blancos'][c]] = ''
    sim = pd.DataFrame(resp, columns=[f'P{i}' for i in claves['item']])
    sim.insert(0, 'id', Zn.index)
    sim.insert(1, 'fecha_aplicacion', cond['fecha_aplicacion'].values)
    sim.insert(2, 'modalidad', cond['modalidad'].values)
    verdad = pd.DataFrame({'id': np.repeat(Zn.index, 7), 'area': CODIGOS * n,
                           'I_verdad': I_v.ravel(), 'R_verdad': R_v.ravel(), 'A_verdad': A_v.ravel()})
    verdad = verdad.merge(cond[['id', 'aleatorio']], on='id')
    return sim, verdad

In [ ]:
# ── 3B.3 Publicar el escenario realista (un solo archivo; se reemplaza en cada ejecución) ──
sim, verdad_pub = simular(ESCENARIO_PUBLICADO)
id_original = dict(zip(llave['id'], llave['nombre'].astype(str)))
publicar = sim.rename(columns={'id': 'Student ID (System)'})
publicar['Student ID (System)'] = publicar['Student ID (System)'].map(id_original)
publicar.to_csv(f'{BASE}/01_entrada/nucleo/chaside/chaside_AAI-sim500_2026-10.csv', index=False, encoding='utf-8')
verdad_pub.to_csv(f'{BASE}/04_procesado/privado/simulacion_chaside_verdad.csv', index=False)
with open(f'{BASE}/04_procesado/privado/simulacion_chaside_parametros.json', 'w') as fh:
    json.dump({'escenario': ESCENARIO_PUBLICADO, **ESCENARIOS[ESCENARIO_PUBLICADO], 'alfa_notas': ALFA_NOTAS_OBS}, fh, indent=2)
print(f'✓ Escenario «{ESCENARIO_PUBLICADO}» publicado en 01_entrada/nucleo/chaside/chaside_AAI-sim500_2026-10.csv')
print('✓ Verdad y parámetros en 04_procesado/privado/ (ocultos al modelo)')

In [ ]:
# ── 3B.4 Control de plausibilidad del archivo publicado ──────────────────────
si = (sim[[f'P{i}' for i in claves['item']]] == 'Sí').values
punt_sim = pd.DataFrame({f'{a}_{t[:3]}': si[:, ((claves['area'] == a) & (claves['tipo'] == t)).values].sum(1)
                         for a in CODIGOS for t in ['interes', 'aptitud']})
print(f"% de 'Sí' = {si.mean() * 100:.0f}% · respondientes al azar = {verdad_pub.drop_duplicates('id')['aleatorio'].sum()}")
print('Puntaje medio (interés 0-10 · aptitud 0-4):')
display(punt_sim.mean().round(1).to_frame('media').T)
v = verdad_pub
print('Correlaciones verdaderas en la simulación · interés–rendimiento: '
      f"{v['I_verdad'].corr(v['R_verdad']):.2f} · autopercepción–rendimiento: {v['A_verdad'].corr(v['R_verdad']):.2f}")

### ✅ Aprobación Módulo 3B
- [ ] Modelo generativo con el rendimiento anclado en las notas, la autopercepción sesgada y el interés poco relacionado con el rendimiento.
- [ ] Un solo archivo publicado (escenario realista). Los escenarios de estrés existen solo en memoria, para probar.
- [ ] Queda claro que es **simulación para pruebas**: en producción habrá un único CHASIDE real por estudiante.

---
# MÓDULO 4 · Adaptador CHASIDE (núcleo): de 98 respuestas a evidencia por área

**Qué resuelve:** convierte las respuestas Sí/No de cada estudiante en dos evidencias por área:

- **interés**, la medida principal del CHASIDE;
- **autopercepción de aptitud**, una evidencia secundaria que se **contrasta con las notas**.

Además mide **cuán confiable es la prueba** con los propios datos y detecta **aplicaciones de baja calidad**.

| Paso | Qué hace | Criterio |
|---|---|---|
| 4.1 | Carga y seudonimiza | Usa la misma llave del Módulo 3, así el CHASIDE y las notas del mismo estudiante se unen |
| 4.2 | Puntaje por área | Proporción de "Sí" **sobre los ítems respondidos**. Un blanco no cuenta como "No" (ausencia ≠ cero) |
| 4.3 | Confiabilidad real (KR-20) | Reemplaza el α provisional del Módulo 1 |
| 4.4 | Criterios de calidad por aplicación | Antigüedad, modalidad, completitud y consistencia → alimentan la rúbrica |
| 4.5 | Escala z | Contra la cohorte, igual que las notas |

Cada paso es una **función**. Así el mismo procedimiento se aplica al CHASIDE publicado y a los escenarios de estrés del Módulo 5.F.

In [ ]:
# ── 4.1 Carga y seudonimización del CHASIDE publicado ─────────────────────────
archivos_ch = sorted(glob.glob(f'{BASE}/01_entrada/nucleo/chaside/*.csv'))
def leer_chaside(ruta):
    d = pd.read_csv(ruta, dtype=str)
    return d.rename(columns={d.columns[0]: '_id'})           # primera columna = identificador del estudiante

ch_raw = pd.concat([leer_chaside(a) for a in archivos_ch], ignore_index=True)
col_id = '_id'
llave['nombre'] = llave['nombre'].astype(str)
ya_seudonimo = set(llave['id'])                             # pruebas registradas desde la interfaz ya vienen con E-xxxxxx
nuevos = [x for x in ch_raw[col_id].unique() if x not in set(llave['nombre']) and x not in ya_seudonimo]
for x in nuevos:                                            # estudiantes con CHASIDE pero sin notas
    nid = f'E-{secrets.token_hex(3)}'
    while nid in set(llave['id']): nid = f'E-{secrets.token_hex(3)}'
    llave.loc[len(llave)] = [x, nid]
llave.to_csv(RUTA_LLAVE, index=False)
mapa = {**dict(zip(llave['nombre'], llave['id'])), **{i: i for i in ya_seudonimo}}
ch = ch_raw.assign(id=ch_raw[col_id].map(mapa)).drop(columns=[col_id])
if 'fecha_aplicacion' in ch:                                # si un estudiante repitió la prueba, vale la MÁS RECIENTE
    ch = ch.sort_values('fecha_aplicacion', kind='stable', na_position='first')
ch = ch.drop_duplicates('id', keep='last')
print(f'{len(archivos_ch)} archivo(s) · {len(ch)} pruebas · {ch.id.isin(rend.id).sum()} con notas (núcleo completo) · {len(nuevos)} ID nuevos')

### 4.2 Puntaje por área

Para el estudiante $e$, el área $a$ y la escala $k$ (interés: 10 ítems; autopercepción de aptitud: 4 ítems):

$$p_{e,a,k} = \frac{\#\,\text{Sí}}{\#\,\text{ítems respondidos}}$$

**Por qué una proporción y no una suma:** si un estudiante deja ítems en blanco, la suma lo castiga como si hubiera dicho "No". La proporción usa solo lo que respondió. La falta de respuestas se maneja aparte, como criterio de calidad (completitud) que **reduce el peso** de la prueba.

In [ ]:
# ── 4.2 Puntaje por área y escala ──────────────────────────────────────────
MAP_SI = {'sí': 1, 'si': 1, '1': 1, 'true': 1, 'x': 1, 'no': 0, '0': 0, 'false': 0}
ITEMS = [f'P{i}' for i in range(1, 99)]

def puntajes(ch):
    resp = ch[ITEMS].apply(lambda s: s.astype(str).str.strip().str.lower().map(MAP_SI)).astype(float)   # blanco → NaN
    resp.index = ch['id'].values
    punt = pd.DataFrame({(a, t): resp[[f'P{i}' for i in claves.query('area == @a and tipo == @t')['item']]].mean(axis=1)
                         for a in CODIGOS for t in ['interes', 'aptitud']})
    return resp, punt

resp, punt = puntajes(ch)
print('Proporción media de "Sí" por área:')
display(punt.mean().unstack().reindex(CODIGOS).round(2).T)

### 4.3 Confiabilidad medida con los datos (KR-20)

$$KR_{20} = \frac{k}{k-1}\left(1 - \frac{\sum_j p_j (1-p_j)}{\sigma^2_{\text{total}}}\right)$$

- **Por qué importa:** en el Módulo 1 el α de CHASIDE era un **supuesto**. Aquí se **mide por área** con la cohorte, y el Módulo 5 usa estos valores.
- **Qué esperar:** las escalas de aptitud tienen solo 4 ítems, así que su KR-20 será menor.

In [ ]:
# ── 4.3 KR-20 por área y escala ────────────────────────────────────────────
def kr20(X):
    X = X.dropna()                                        # solo quien respondió todos los ítems de la escala
    k, p = X.shape[1], X.mean()
    return k / (k - 1) * (1 - (p * (1 - p)).sum() / X.sum(axis=1).var(ddof=1))

def confiabilidad(resp):
    filas = [{'area': a, 'tipo': t, 'KR20': round(kr20(resp[[f'P{i}' for i in claves.query('area == @a and tipo == @t')['item']]]), 3)}
             for a in CODIGOS for t in ['interes', 'aptitud']]
    conf = pd.DataFrame(filas)
    return conf, {(r.area, r.tipo): float(np.clip(r.KR20, 0.30, 0.95)) for r in conf.itertuples()}

conf, ALFA_CH = confiabilidad(resp)
display(conf.pivot(index='tipo', columns='area', values='KR20').reindex(columns=CODIGOS))
m = conf.groupby('tipo')['KR20'].mean()
print(f"KR-20 medio · interés {m['interes']:.2f} (supuesto: 0.80) · aptitud {m['aptitud']:.2f} (supuesto: 0.60)")

### 4.4 Criterios de calidad de cada aplicación (para la rúbrica del Módulo 1)

| Criterio | Cómo se mide | Niveles |
|---|---|---|
| **Antigüedad** | `fecha_aplicacion` frente a la fecha de referencia | < 1 año · 1–2 años · > 2 años |
| **Modalidad** | Columna `modalidad` | supervisada · autónoma |
| **Completitud** | % de los 98 ítems respondidos | ≥ 95 % · 80–95 % · < 80 % |
| **Consistencia** | **Perfil plano**: quien responde al azar da ~50 % de "Sí" en todas las áreas | Alerta si la dispersión del perfil está bajo el percentil 5 **y** la tasa de "Sí" está entre 35 % y 65 % |

> El detector de consistencia es una **heurística explicable**. En la simulación se mide su precisión (Módulo 5).

In [ ]:
# ── 4.4 Criterios por aplicación ───────────────────────────────────────────
FECHA_REF = pd.Timestamp('2026-10-01')

def criterios(ch, resp, punt):
    meta = ch.set_index('id')
    edad = (FECHA_REF - pd.to_datetime(meta['fecha_aplicacion'], errors='coerce')).dt.days / 30.44 \
           if 'fecha_aplicacion' in meta else pd.Series(np.nan, index=meta.index)
    comp = resp.notna().mean(axis=1)
    crit = pd.DataFrame(index=punt.index)
    crit['antiguedad'] = np.where(edad.isna().values, 'menos_1_anio',
                         np.select([edad.values < 12, edad.values < 24], ['menos_1_anio', '1_a_2_anios'], 'mas_2_anios'))
    crit['modalidad'] = meta['modalidad'].fillna('supervisada').values if 'modalidad' in meta else 'supervisada'
    crit['completitud'] = np.select([comp >= .95, comp >= .80], ['completa', 'parcial'], 'incompleta')
    disp = punt.xs('interes', axis=1, level=1).std(axis=1)
    crit['consistencia'] = np.where((disp < disp.quantile(0.05)) & resp.mean(axis=1).between(.35, .65), 'alerta', 'sin_alertas')
    crit['completitud_%'] = (comp * 100).round(0)
    return crit

crit = criterios(ch, resp, punt)
for c in ['antiguedad', 'modalidad', 'completitud', 'consistencia']:
    print(f'{c:13s}', crit[c].value_counts().to_dict())

### 4.5 Escala z y resultado del adaptador

$$z^{\,int}_{e,a} = \frac{p^{\,int}_{e,a} - \bar p^{\,int}_a}{s^{\,int}_a} \qquad z^{\,apt}_{e,a} = \frac{p^{\,apt}_{e,a} - \bar p^{\,apt}_a}{s^{\,apt}_a}$$

**Resultado:** una fila por estudiante y área, con interés (z), autopercepción de aptitud (z), confiabilidad del área y criterios de calidad. Es la entrada del modelo bayesiano.

In [ ]:
# ── 4.5 Estandarización + función completa del adaptador ─────────────────────
def evidencia_chaside(punt, crit, ALFA):
    z = (punt - punt.mean()) / punt.std()
    ev = pd.concat([z.xs('interes', axis=1, level=1).stack().rename('z_int'),
                    z.xs('aptitud', axis=1, level=1).stack().rename('z_apt')], axis=1).rename_axis(['id', 'area']).reset_index()
    ev = ev.merge(crit.rename_axis('id').reset_index(), on='id')
    ev['alfa_int'] = [ALFA[(a, 'interes')] for a in ev['area']]
    ev['alfa_apt'] = [ALFA[(a, 'aptitud')] for a in ev['area']]
    return ev

def procesar_chaside(ch):
    # Adaptador completo: respuestas → evidencia por área (se reutiliza en los escenarios de estrés)
    resp, punt = puntajes(ch)
    conf, ALFA = confiabilidad(resp)
    return evidencia_chaside(punt, criterios(ch, resp, punt), ALFA), conf, ALFA

ev_ch = evidencia_chaside(punt, crit, ALFA_CH)
ev_ch.to_csv(f'{BASE}/04_procesado/chaside_evidencia.csv', index=False)
conf.to_csv(f'{BASE}/04_procesado/confiabilidad_chaside.csv', index=False)
print('✓ 04_procesado/chaside_evidencia.csv · confiabilidad_chaside.csv')
display(ev_ch.head(7).round(2))

### ✅ Aprobación Módulo 4
- [ ] Puntaje como proporción sobre los ítems respondidos (los blancos no cuentan como "No").
- [ ] Confiabilidad **medida** con KR-20 por área.
- [ ] Criterios de calidad por aplicación, calculados automáticamente.
- [ ] Todo empaquetado en funciones, reutilizables para el CHASIDE real y para las pruebas de estrés.

---
# MÓDULO 5 · Modelo bayesiano: Interés × Rendimiento por área

Este módulo responde cinco preguntas, cada una en su sección:

| # | Pregunta | Sección |
|---|---|---|
| 1 | ¿Qué hace un algoritmo bayesiano? | 5.A |
| 2 | ¿Cómo lo estamos cumpliendo en este proyecto? | 5.B |
| 3 | ¿Cómo funciona con datos de entrenamiento? | 5.C |
| 4 | ¿Qué resultados obtenemos y cómo se interpretan? | 5.D |
| 5 | ¿Cómo evaluamos la fidelidad del resultado? | 5.E y 5.F (estrés) |

---
## 5.A · ¿Qué hace un algoritmo bayesiano?

Un algoritmo bayesiano **actualiza una creencia con evidencia**. Parte de lo que se sabe antes de mirar los datos del individuo (el **prior**), observa evidencias con distinto grado de confiabilidad (la **verosimilitud**) y produce una creencia actualizada con su margen de error (el **posterior**):

$$\underbrace{P(\theta \mid \text{datos})}_{\text{posterior}} \;=\; \frac{\overbrace{P(\text{datos} \mid \theta)}^{\text{verosimilitud}}\;\cdot\;\overbrace{P(\theta)}^{\text{prior}}}{P(\text{datos})}$$

- $\theta$ es lo que queremos conocer y **no se observa directamente**: el interés real y el rendimiento real de un estudiante en un área.
- Los **datos** son lo observable: las notas y las respuestas al CHASIDE. Son mediciones **imperfectas** de $\theta$.

**Analogía:** una consejera del DECE parte de lo típico del grupo, mira las notas (mucha confianza: son años de evidencia), mira el test (confía según su calidad) y llega a una opinión con un grado de seguridad.

**Tres propiedades que lo distinguen de un clasificador común (por ejemplo, un árbol de decisión):**

1. **Entrega incertidumbre, no solo un valor**: "interés en Salud P82 ± margen", no "Salud: sí".
2. **Combina fuentes según su confiabilidad**, con una regla matemática y no con pesos puestos a mano.
3. **Funciona con datos incompletos**: si falta una fuente, el posterior queda más incierto, sin imputar ni descartar al estudiante.

---
## 5.B · ¿Cómo lo estamos cumpliendo? La red bayesiana del proyecto (decisión B)

Para cada área CHASIDE $a$, una **red bayesiana gaussiana lineal** con 2 variables latentes y 3 evidencias:

```
                       ρ_a  (aprendida de los datos)
        ┌───────────────────────────────────────────┐
        │                                           │
  [ Interés I_a ]                        [ Rendimiento R_a ]          ← LATENTES  (prior: N(0, Σ₀))
        │                                  ║                 │
        ▼                                  ▼                 ▼
  CHASIDE interés                       NOTAS          CHASIDE aptitud   ← EVIDENCIAS
  (+ IPP-J u otras, opcionales)        (ANCLA)        (autopercepción, secundaria)
```

**Decisión B · el rendimiento se ancla en las notas.**

- "Rendimiento" significa **desempeño académico real** en las materias del área, y las notas son su medida directa.
- La aptitud que el estudiante declara en el CHASIDE es **autopercepción**: aporta, pero el modelo aprende cuánto se parece a sus notas y la pondera en consecuencia.
- La **diferencia** entre autopercepción y notas se reporta aparte, como información para el DECE.

**1 · Prior:** sin evidencia, el estudiante está en el promedio de su cohorte; interés y rendimiento pueden estar correlacionados ($\rho_a$):

$$\begin{pmatrix} I_a \\ R_a \end{pmatrix} \sim N\!\left(\mathbf{0},\; \Sigma_0 = \begin{pmatrix}1 & \rho_a\\ \rho_a & 1\end{pmatrix}\right)$$

**2 · Verosimilitud:** cada evidencia $k$ es la latente más un error:

$$y_k = H_k \begin{pmatrix} I_a \\ R_a \end{pmatrix} + \varepsilon_k, \qquad \varepsilon_k \sim N(0, \sigma^2_k), \qquad \sigma^2_k = \frac{1-\alpha_k}{\alpha_k}\prod f_k$$

| Evidencia | Mide | $\alpha_k$ (cuánto se parece a la latente) |
|---|---|---|
| CHASIDE interés | $I_a$ | KR-20 del área (Módulo 4) |
| **Notas** | $R_a$ | **Consistencia de las notas entre años** (aprendida, 5.C) |
| CHASIDE autopercepción | $R_a$ | $\lambda^2$: cuánto coincide la autopercepción con las notas (aprendida, 5.C) |

Los factores $f_k$ son los criterios de calidad de *esa aplicación* (rúbrica del Módulo 1).

**3 · Posterior** (solución exacta, sin aproximaciones):

$$\Lambda = \Sigma_0^{-1} + \sum_k \frac{H_k^\top H_k}{\sigma_k^2} \qquad \Sigma_{post} = \Lambda^{-1} \qquad \mu_{post} = \sum_k \underbrace{\Sigma_{post}\,\frac{H_k^\top y_k}{\sigma_k^2}}_{\text{aporte de la fuente }k}$$

**Explicabilidad:** $\mu_{post}$ es una **suma de aportes por fuente**. Por ejemplo: *"tu rendimiento en Exactas es +1.1: tus notas aportan +0.9 y tu autopercepción +0.2"*.

**Relación con el Módulo 1:** con una sola latente y ρ = 0, esta fórmula es exactamente la ponderación por precisión del Módulo 1. Aquí se **generaliza**.

In [ ]:
# ── 5.1 Ensamblar la evidencia del núcleo ──────────────────────────────────
from dataclasses import replace
from scipy.stats import multivariate_normal

notas_ev = rend[['id', 'area', 'R', 'W_efectiva']].copy()
notas_ev['z_notas'] = notas_ev.groupby('area')['R'].transform(lambda s: (s - s.mean()) / s.std())

def ensamblar(ev_ch):
    ids = sorted(set(ev_ch['id']) & set(notas_ev['id']))
    return ev_ch[ev_ch['id'].isin(ids)].merge(notas_ev[['id', 'area', 'z_notas', 'W_efectiva']], on=['id', 'area'], how='left'), ids

ev, ids_nucleo = ensamblar(ev_ch)
print(f'Estudiantes con núcleo completo (notas + CHASIDE): {len(ids_nucleo)} · filas estudiante×área: {len(ev)}')

---
## 5.C · ¿Cómo funciona con datos de entrenamiento?

**"Entrenar"** aquí significa **estimar con los datos los parámetros que el Módulo 1 tenía como supuestos**. Es aprendizaje **no supervisado**: no necesita conocer la "respuesta correcta" de ningún estudiante.

| Parámetro | Supuesto (Módulo 1) | Cómo se aprende con datos |
|---|---|---|
| Confiabilidad del CHASIDE ($\alpha_{int}$) | 0.80 | **KR-20** por área (Módulo 4) |
| Confiabilidad de las **notas** ($\alpha_{notas}$) | 0.85 con penalización ×2.5 | **Consistencia entre años**: correlación media del rendimiento del área entre 9.º, 10.º y 11.º, llevada a $k$ años con Spearman-Brown: $\alpha = \frac{k\,\bar r}{1+(k-1)\bar r}$ |
| Fidelidad de la **autopercepción** ($\lambda^2$) | 0.60 | **Desatenuación**: $\lambda = r(\text{autopercepción}, \text{notas}) / \sqrt{\alpha_{notas}}$ |
| Correlación interés–rendimiento ($\rho_a$) | 0 | $\rho_a = r(\text{interés}, \text{notas}) / \sqrt{\alpha_{int}\,\alpha_{notas}}$ |

**Partición 70 / 30:** los parámetros se aprenden **solo con el 70 %** de los estudiantes, y el modelo se evalúa con el **30 % que no vio** (5.E). La verdad simulada **no se usa para entrenar**.

**Con más datos en el tiempo:**

- Cada cohorte nueva re-estima los parámetros con más estudiantes, y se vuelven más estables.
- Para un mismo estudiante, el **posterior de este año puede usarse como prior del siguiente**: la actualización longitudinal que pide la propuesta.

In [ ]:
# ── 5.2 Entrenamiento: partición y aprendizaje de parámetros ───────────────
def particion(ids, semilla=7):
    arr = np.array(ids); np.random.default_rng(semilla).shuffle(arr)
    k = int(0.7 * len(arr)); return set(arr[:k]), set(arr[k:])

def aprender(ev, train, ALFA):
    alfa_n = confiabilidad_notas(por_anio, train)
    tr = ev[ev['id'].isin(train)]
    P = {}
    for a in CODIGOS:
        d = tr[tr['area'] == a].dropna(subset=['z_int', 'z_apt', 'z_notas'])
        an, ai = alfa_n[a], ALFA[(a, 'interes')]
        r_an, r_in = d['z_apt'].corr(d['z_notas']), d['z_int'].corr(d['z_notas'])
        lam = float(np.clip(r_an / np.sqrt(an), 0.10, 0.90))
        P[a] = {'alfa_notas': an, 'lam_apt': lam, 'r_apt_notas': r_an,
                'rho': float(np.clip(r_in / np.sqrt(ai * an), -0.9, 0.9)),
                'W_med': float(ev.loc[ev['area'] == a, 'W_efectiva'].median())}
    return P

TRAIN, TEST = particion(ids_nucleo)
PARAM = aprender(ev, TRAIN, ALFA_CH)
tabla_param = pd.DataFrame({a: {'KR20 interés': ALFA_CH[(a, 'interes')],
                                'α notas (supuesto 0.69)': p['alfa_notas'],
                                'λ² autopercepción (supuesto 0.60)': p['lam_apt']**2,
                                'ρ interés–rendimiento (supuesto 0)': p['rho']} for a, p in PARAM.items()}).T.round(2)
print(f'Entrenamiento: {len(TRAIN)} estudiantes · Prueba: {len(TEST)} estudiantes')
display(tabla_param)

In [ ]:
# ── 5.3 Motor de inferencia (posterior exacto por estudiante y área) ────────
NOTAS_APR = replace(NOTAS, criterios={'validacion': 'publicada', 'equivalencia': 'directa'})  # α aprendido ya incorpora la validez

def posterior_area(r, p, modo='aprendido', usar=('int', 'apt', 'notas')):
    rho = p['rho'] if modo == 'aprendido' else 0.0
    Lam, b, obs = np.linalg.inv(np.array([[1, rho], [rho, 1]])), np.zeros(2), []
    crit_ap = {k: r[k] for k in ['antiguedad', 'modalidad', 'completitud', 'consistencia']}
    if 'int' in usar and pd.notna(r['z_int']):
        _, s2, _ = sigma2_efectiva(replace(CHASIDE_INT, alfa=r['alfa_int']), crit_ap)
        obs.append(('CHASIDE interés', np.array([1., 0.]), r['z_int'] / np.sqrt(r['alfa_int']), s2))
    if 'notas' in usar and pd.notna(r['z_notas']):
        a_n, fuente = (p['alfa_notas'], NOTAS_APR) if modo == 'aprendido' else (0.85, NOTAS)
        _, s2, _ = sigma2_efectiva(replace(fuente, alfa=a_n), {})
        f_w = float(np.clip(p['W_med'] / r['W_efectiva'], 0.5, 3.0)) if pd.notna(r['W_efectiva']) else 1.0
        obs.append(('Notas', np.array([0., 1.]), r['z_notas'] / np.sqrt(a_n), s2 * f_w))
    if 'apt' in usar and pd.notna(r['z_apt']):
        lam = p['lam_apt'] if modo == 'aprendido' else np.sqrt(0.60)
        _, s2, _ = sigma2_efectiva(replace(CHASIDE_APT, alfa=lam**2), crit_ap)
        obs.append(('Autopercepción', np.array([0., 1.]), r['z_apt'] / lam, s2))
    for _, H, y, s2 in obs:
        Lam += np.outer(H, H) / s2; b += H * y / s2
    Sig = np.linalg.inv(Lam)
    return Sig @ b, Sig, {nom: Sig @ (H * y / s2) for nom, H, y, s2 in obs}

def probabilidades_cuadrante(mu, Sig):
    sI, sR = np.sqrt(np.diag(Sig))
    pI, pR = norm.cdf(mu[0] / sI), norm.cdf(mu[1] / sR)
    pAA = multivariate_normal(mean=-mu, cov=Sig).cdf([0, 0])          # P(I>0, R>0) con la correlación del posterior
    return {CUADRANTES[(True, True)]: pAA, CUADRANTES[(True, False)]: pI - pAA,
            CUADRANTES[(False, True)]: pR - pAA, CUADRANTES[(False, False)]: 1 - pI - pR + pAA}

def inferir(ev, P, modo='aprendido', usar=('int', 'apt', 'notas'), con_cuadrantes=True):
    filas = []
    for r in ev.to_dict('records'):
        p = P[r['area']]
        mu, Sig, aportes = posterior_area(r, p, modo, usar)
        f = {'id': r['id'], 'area': r['area'], 'mu_I': mu[0], 'sd_I': np.sqrt(Sig[0, 0]), 'mu_R': mu[1], 'sd_R': np.sqrt(Sig[1, 1])}
        for nom, c in aportes.items():
            f[f'aporte_I · {nom}'], f[f'aporte_R · {nom}'] = c[0], c[1]
        # Brecha de autopercepción: lo que el estudiante cree de sí vs. lo que sus notas predicen
        if pd.notna(r['z_apt']) and pd.notna(r['z_notas']):
            f['brecha_auto'] = (r['z_apt'] - p['r_apt_notas'] * r['z_notas']) / np.sqrt(1 - p['r_apt_notas']**2)
        if con_cuadrantes:
            pc = probabilidades_cuadrante(mu, Sig)
            mejor = max(pc, key=pc.get)
            f.update({f'P[{k}]': v for k, v in pc.items()})
            f['cuadrante'] = mejor if pc[mejor] >= UMBRAL_CERTEZA else '❔ Indeterminado: falta evidencia'
            f['certeza'] = pc[mejor]
        filas.append(f)
    out = pd.DataFrame(filas)
    out['pct_I'], out['pct_R'] = z_a_percentil(out['mu_I']).round(0), z_a_percentil(out['mu_R']).round(0)
    if 'brecha_auto' in out:
        out['autopercepcion'] = np.select([out['brecha_auto'] > 1, out['brecha_auto'] < -1],
                                          ['se percibe por ENCIMA de sus notas', 'se percibe por DEBAJO de sus notas'], 'coherente con sus notas')
    out['conjunto'] = np.where(out['id'].isin(TRAIN), 'entrenamiento', 'prueba')
    return out

post = inferir(ev, PARAM)
print(f'Posterior calculado para {post.id.nunique()} estudiantes × 7 áreas = {len(post)} filas')

---
## 5.D · ¿Qué resultados obtenemos y cómo se interpretan?

| Resultado | Qué significa | Cómo leerlo |
|---|---|---|
| Interés (percentil ± σ) | Interés estimado comparado con la cohorte | P80 = más interés que el 80 % del grupo. σ cerca de 1 = casi no sabemos; cerca de 0.3 = bastante seguro |
| Rendimiento (percentil ± σ) | Desempeño académico en las materias del área | Se sostiene sobre todo en las notas |
| Zona del plano y certeza | La zona más probable, **solo si supera el 60 %** | Si no, "❔ Indeterminado" |
| Autopercepción | Si el estudiante se ve a sí mismo por encima, por debajo o igual que sus notas | Muy útil para conversar: un talento que el estudiante no ve, o una expectativa que conviene revisar |
| Aportes | Cuánto empujó cada fuente el resultado | Explica el *porqué* |

**Reglas de interpretación para el DECE, docentes y familias:**

1. **No es un diagnóstico.** Es una estimación con margen de error que abre la conversación.
2. **Lea la certeza junto al valor.** 61 % es una hipótesis; 95 % es un patrón claro.
3. **🔵 Talento no explorado** y **🟡 Interés a fortalecer** son las zonas más útiles para orientar, porque muestran una tensión entre lo que gusta y lo que rinde.
4. **"Indeterminado" no es un error**: indica qué evidencia falta.

In [ ]:
# ── 5.4 Lectura de un estudiante (ejemplo del conjunto de prueba) ──────────
def reporte_estudiante(id_est):
    p = post[post['id'] == id_est].set_index('area').reindex(CODIGOS)
    e = ev[ev['id'] == id_est].iloc[0]
    print(f'▶ Estudiante {id_est} ({p.conjunto.iloc[0]}) · CHASIDE: {e.antiguedad}, {e.modalidad}, {e.completitud}, consistencia {e.consistencia}')
    print(f"{'Área':30s} {'Interés':>11s} {'Rendimiento':>12s}   {'Zona (certeza)':38s} {'Autopercepción':36s} Aportes al rendimiento")
    for a, r in p.iterrows():
        ap = ' · '.join(f"{k.split('· ')[1]} {v:+.2f}" for k, v in r.items() if str(k).startswith('aporte_R') and pd.notna(v) and abs(v) > .005)
        print(f"{a} {EJES[a][:27]:27s}  P{r.pct_I:>3.0f}±{r.sd_I:.2f}  P{r.pct_R:>3.0f}±{r.sd_R:.2f}   "
              f"{r.cuadrante:32s}({r.certeza:.0%})  {r.autopercepcion:36s} {ap}")

reporte_estudiante(sorted(TEST)[0])

In [ ]:
# ── 5.5 Resultados de la cohorte ───────────────────────────────────────────
print('Distribución de zonas (todas las áreas, todos los estudiantes):')
display((post['cuadrante'].value_counts(normalize=True) * 100).round(1).rename('% de áreas').to_frame())
print('Autopercepción frente a notas:')
display((post['autopercepcion'].value_counts(normalize=True) * 100).round(1).rename('% de áreas').to_frame())
print('Incertidumbre media (antes de ver evidencia = 1.00):')
display(post.groupby('area')[['sd_I', 'sd_R']].mean().reindex(CODIGOS).round(2).T)
w = post.filter(like='aporte_R').abs().mean()
print('Peso medio de cada fuente en el RENDIMIENTO (|aporte| promedio):', (w / w.sum() * 100).round(0).to_dict())

---
## 5.E · ¿Cómo evaluamos la fidelidad del resultado?

La simulación guardó el perfil **verdadero** de cada estudiante y el modelo nunca lo vio. Sobre el **30 % de prueba** se compara lo estimado con lo verdadero:

| # | Métrica | Pregunta que responde | Referencia de "bueno" |
|---|---|---|---|
| 1 | **Recuperación** (r y RMSE entre estimado y verdad) | ¿Encuentra el perfil real? | r > 0.7 |
| 2 | **Calibración** (cobertura del intervalo del 90 %) | ¿La incertidumbre que reporta es honesta? | 85–95 % |
| 3 | **Área dominante** (top-1 y top-3 de interés) | ¿Acierta en el área de mayor interés? | Sobre el azar (14 % y 43 %) |
| 4 | **Zonas del plano** (exactitud, % indeterminados, Brier) | ¿Ubica bien cada área? ¿Sus probabilidades son confiables? | Brier < 0.75 (azar) |
| 5 | **Comparación** con los supuestos del Módulo 1, con una sola fuente y con la línea base | ¿Aprender y combinar mejora algo? | Bayes aprendido ≥ resto |
| 6 | **Detector de respuestas al azar** | ¿Encuentra las aplicaciones de mala calidad? | Sensibilidad y precisión |

> **Límite:** esto mide fidelidad frente a la simulación. La validez con estudiantes reales requiere los CHASIDE reales y la valoración del DECE (Módulo 8).

In [ ]:
# ── 5.6 Evaluación de fidelidad sobre el conjunto de PRUEBA ────────────────
def metricas(df, verdad, test):
    d = df[df['id'].isin(test)].merge(verdad, on=['id', 'area'])
    m = {}
    for dim, col in [('I', 'I_verdad'), ('R', 'R_verdad')]:
        m[f'r_{dim}'] = d[f'mu_{dim}'].corr(d[col])
        m[f'RMSE_{dim}'] = np.sqrt(((d[f'mu_{dim}'] - d[col]) ** 2).mean())
        if f'sd_{dim}' in d:
            m[f'cobertura90_{dim}'] = (np.abs(d[col] - d[f'mu_{dim}']) <= 1.645 * d[f'sd_{dim}']).mean()
    top_v = d.loc[d.groupby('id')['I_verdad'].idxmax()].set_index('id')['area']
    top_e = d.loc[d.groupby('id')['mu_I'].idxmax()].set_index('id')['area']
    top3 = d.sort_values('mu_I', ascending=False).groupby('id')['area'].apply(lambda s: set(s[:3]))
    m['top1_interes'] = (top_e == top_v.reindex(top_e.index)).mean()
    m['top1_en_top3'] = np.mean([top_v[i] in top3[i] for i in top3.index])
    if 'cuadrante' in d:
        d['zona_verdad'] = [CUADRANTES[(i > 0, r > 0)] for i, r in zip(d['I_verdad'], d['R_verdad'])]
        det = d[~d['cuadrante'].str.startswith('❔')]
        m['exactitud_zona'] = (det['cuadrante'] == det['zona_verdad']).mean()
        m['indeterminados'] = 1 - len(det) / len(d)
        P = d[[f'P[{z}]' for z in CUADRANTES.values()]].values
        Y = np.array([[z == zv for z in CUADRANTES.values()] for zv in d['zona_verdad']], dtype=float)
        m['brier'] = ((P - Y) ** 2).sum(axis=1).mean()
    return m, d

RUTA_VERDAD = f'{BASE}/04_procesado/privado/simulacion_chaside_verdad.csv'
if os.path.exists(RUTA_VERDAD):
    verdad = pd.read_csv(RUTA_VERDAD)
    ev_test = ev[ev['id'].isin(TEST)]
    filas = {'Bayes aprendido (núcleo completo)': metricas(post, verdad, TEST)[0]}
    for nombre, kw in [('Bayes con supuestos del Módulo 1', {'modo': 'declarado'}),
                       ('Solo CHASIDE (sin notas)', {'usar': ('int', 'apt')}),
                       ('Solo notas para rendimiento', {'usar': ('int', 'notas')})]:
        filas[nombre] = metricas(inferir(ev_test, PARAM, con_cuadrantes=False, **kw), verdad, TEST)[0]
    base = ev[['id', 'area']].assign(mu_I=ev['z_int'], mu_R=ev[['z_apt', 'z_notas']].mean(axis=1))
    filas['Línea base ingenua (promedio simple)'] = metricas(base, verdad, TEST)[0]
    tabla_eval = pd.DataFrame(filas).T
    print('EVALUACIÓN EN EL CONJUNTO DE PRUEBA (escenario publicado)')
    display(tabla_eval.round(3))

    _, d = metricas(post, verdad, TEST)
    d['zona_max'] = d[[f'P[{z}]' for z in CUADRANTES.values()]].idxmax(axis=1).str[2:-1]
    d['tramo'] = pd.cut(d['certeza'], [0, .6, .75, .9, 1.0], labels=['<60% (indet.)', '60-75%', '75-90%', '>90%'])
    print('Calibración de la certeza: ¿cuando el modelo dice X % de certeza, acierta X %?')
    display(d.groupby('tramo', observed=True).apply(lambda g: pd.Series({
        'n': len(g), 'certeza dicha': g['certeza'].mean(), 'acierto real': (g['zona_max'] == g['zona_verdad']).mean()}),
        include_groups=False).round(2).T)

    v_al = verdad.drop_duplicates('id').set_index('id')['aleatorio'].astype(bool)
    alerta = ev.drop_duplicates('id').set_index('id')['consistencia'].eq('alerta')
    vp = (alerta & v_al.reindex(alerta.index).fillna(False)).sum()
    print(f'Detector de respuestas al azar: {v_al.sum()} reales · {alerta.sum()} alertas · '
          f'sensibilidad {vp / max(v_al.sum(), 1):.0%} · precisión {vp / max(alerta.sum(), 1):.0%}')

---
## 5.F · Pruebas de estrés: ¿el modelo resiste datos difíciles?

Se repite **todo el proceso** (simular → adaptar CHASIDE → aprender con el 70 % → inferir → evaluar con el 30 %) en los cuatro escenarios del Módulo 3B, que van del más fácil al más difícil. **Nada de esto se guarda**: es solo para probar el modelo.

**Qué significa "resistir":**

- El **rendimiento** se sigue recuperando bien, porque está anclado en las notas.
- El **interés** se recupera según la calidad del test: en "estrés" baja, y eso es esperable.
- La **calibración se mantiene cerca del 90 %**. Esta es la prueba clave: cuando la evidencia empeora, el modelo debe **reconocer más incertidumbre** (más indeterminados), no inventar seguridad.

In [ ]:
# ── 5.7 Pruebas de estrés (en memoria, no se guardan) ──────────────────────
filas = {}
for nombre in ESCENARIOS:
    sim_e, verdad_e = simular(nombre)
    ev_ch_e, _, ALFA_e = procesar_chaside(sim_e)
    ev_e, ids_e = ensamblar(ev_ch_e)
    tr_e, te_e = particion(ids_e)
    P_e = aprender(ev_e, tr_e, ALFA_e)
    post_e = inferir(ev_e[ev_e['id'].isin(te_e)], P_e)
    m, d_e = metricas(post_e, verdad_e, te_e)
    m['% talento no explorado + interés a fortalecer'] = d_e['cuadrante'].isin([CUADRANTES[(False, True)], CUADRANTES[(True, False)]]).mean()
    m['λ² autopercepción aprendida'] = np.mean([p['lam_apt']**2 for p in P_e.values()])
    m['ρ aprendida'] = np.mean([p['rho'] for p in P_e.values()])
    filas[f"{nombre}{' (publicado)' if nombre == ESCENARIO_PUBLICADO else ''}"] = m
    print(f'✓ {nombre}')
tabla_estres = pd.DataFrame(filas).T[['r_I', 'r_R', 'cobertura90_I', 'cobertura90_R', 'top1_en_top3', 'exactitud_zona', 'indeterminados',
                                      'brier', '% talento no explorado + interés a fortalecer', 'λ² autopercepción aprendida', 'ρ aprendida']]
display(tabla_estres.round(3))

### Cómo leer las pruebas de estrés

- **r_R estable en todos los escenarios** → el rendimiento depende de las notas, no del test. Es la decisión B funcionando.
- **r_I baja cuando el test es peor** (estrés) → es esperable: el interés solo lo mide el test. Con datos reales, sumar fuentes de interés (IPP-J, entrevistas) es la forma de reforzar este eje.
- **Cobertura cerca del 90 % en todos los escenarios** → el modelo es **honesto**: si la evidencia empeora, lo reconoce.
- **Más indeterminados en estrés** → el modelo prefiere decir "no sé" antes que equivocarse con seguridad.
- **El modelo aprende el escenario**: λ² y ρ aprendidos bajan a medida que los datos se desalinean. Detecta solo cuánto confiar en la autopercepción.

In [ ]:
# ── 5.8 Guardar resultados (una sola versión, se reemplaza en cada ejecución) ──
SAL = f'{BASE}/05_salidas'
post.to_csv(f'{SAL}/posterior_por_area.csv', index=False)
tabla_param.to_csv(f'{SAL}/parametros_aprendidos.csv')
if 'tabla_eval' in globals(): tabla_eval.to_csv(f'{SAL}/evaluacion_fidelidad.csv')
tabla_estres.to_csv(f'{SAL}/evaluacion_estres.csv')
print('✓ 05_salidas/: posterior_por_area · parametros_aprendidos · evaluacion_fidelidad · evaluacion_estres')

### ✅ Aprobación Módulo 5
- [ ] Decisión B: el rendimiento se ancla en las notas; la autopercepción es secundaria y su diferencia con las notas se reporta.
- [ ] Inferencia exacta con aportes por fuente.
- [ ] Entrenamiento no supervisado (70 %) y evaluación con el 30 % que el modelo no vio.
- [ ] Interpretación: percentil ± incertidumbre, zona con certeza mínima del 60 %, autopercepción.
- [ ] Pruebas de estrés en 4 escenarios: el modelo mantiene la calibración y reconoce más incertidumbre cuando los datos empeoran.

---
# MÓDULO 7 · Carreras: afinidad, probabilidad de top 5 y razones

**Qué resuelve:** convierte el perfil por área (Módulo 5) en una **lista de carreras ordenada y explicada**, usando la tabla de carreras del país (`03_referencias/carreras/carreras_EC.csv`). La tabla es **reemplazable**: otro país, otra lista, el mismo código.

**Qué NO dice el sistema:** "Pedro es 90 % médico". Ningún modelo puede afirmar eso con estos datos, porque no tenemos información de qué carreras eligieron y terminaron estudiantes anteriores.

**Qué SÍ dice, con honestidad:**

| Resultado | Pregunta que responde | Ejemplo |
|---|---|---|
| **Afinidad (0–100)** | ¿Cuánto se parece el perfil del estudiante a lo que pide la carrera? | Medicina 78 |
| **P(top 5)** | Con la incertidumbre del modelo, ¿qué tan seguro es que esta carrera esté entre sus 5 más afines? | 0.84 → aparece en el top 5 en el 84 % de los escenarios plausibles |
| **Razón** | ¿Por qué aparece? | "Salud (pesa 5/5 en la carrera): interés P82, rendimiento P75 · materias: Biología 9.4" |

### 7.A · La fórmula de afinidad

Cada carrera tiene un perfil CHASIDE de 1 a 5 (cuánto pide de cada área). Se convierte en pesos: un 1 ("no lo pide") vale 0 y un 5 ("lo pide mucho") vale 1:

$$w_{c,a} = \left(\frac{p_{c,a} - 1}{4}\right)^{\gamma} \qquad \tilde w_{c,a} = \frac{w_{c,a}}{\sum_{a'} w_{c,a'}}$$

El exponente $\gamma$ lo elige el algoritmo (ver 7.A2). Por ejemplo, con $\gamma = 3$: un 5 pesa 1, un 4 pesa 0.42, un 3 pesa 0.13 y un 2 pesa 0.02.

El estudiante aporta, en cada área, su **interés** y su **rendimiento** expresados como percentil (0–1):

$$s_a = \beta_I\,\Phi(\mu_{I,a}) + \beta_R\,\Phi(\mu_{R,a}) \qquad \beta_I + \beta_R = 1$$

$$x_c = \sum_a \tilde w_{c,a}\, s_a \qquad \text{Afinidad}_c = 100\;\Phi\!\left(\frac{x_c - \bar x_c}{s_{x_c}}\right)$$

$\bar x_c$ y $s_{x_c}$ son la media y la desviación de $x_c$ en la cohorte. La afinidad se lee así: **"encaja en esta carrera mejor que el 85 % de sus compañeros"**.

**Por qué así:**

1. **Promedio ponderado por lo que pide la carrera.** Para Medicina, Salud (5) pesa mucho más que Artes (1, peso 0). Una carrera no "castiga" por áreas que no necesita.
2. **Interés y rendimiento juntos.** Una carrera conviene cuando gusta **y** se tiene base académica. Por defecto **β_I = β_R = 0.5** (50/50), como acordamos. El DECE puede moverlo, y 7.4 muestra cuánto cambia el resultado.
3. **Comparado con la cohorte, carrera por carrera.** Así todas las carreras compiten en igualdad (ver 7.A2) y 50 significa "perfil promedio para esa carrera".

### 7.A2 · Equilibrio: que ninguna carrera acapare el ranking

**El problema que apareció:** con la fórmula simple, *Artes Visuales* y *Artes Musicales* salían en el top 3 de 1 de cada 4 estudiantes. Su perfil pide 5 en Artísticas y 1 en todo lo demás, mientras que una ingeniería pide 5 en Ingenierías **y** 4, 3, 2 en otras áreas. Un promedio de muchas áreas casi nunca es extremo; el de una sola área sí. Resultado: las carreras "de un solo pico" ganaban, y las ingenierías y las de Defensa quedaban abajo aunque el estudiante fuera fuerte ahí.

**La corrección (sin tocar la tabla a mano), en dos pasos:**

1. **Comparar con la cohorte, carrera por carrera** (la estandarización de arriba): cada carrera tiene la misma escala.
2. **Afinar los pesos con $\gamma$:** que el área principal de la carrera mande y las secundarias sumen sin diluirla.

**Cómo elige $\gamma$ el algoritmo:** prueba $\gamma$ = 1, 1.5, 2, 3 y 4. Para cada valor mide si el área de la carrera #1 de cada estudiante se reparte igual que su área más fuerte (distancia de variación total, TV). Elige el **menor $\gamma$ con TV ≤ 0.10**: el más equilibrado que todavía conserva lo que aportan las áreas secundarias.

### 7.B · ¿De dónde sale P(top 5)? (Monte Carlo sobre el posterior)

El Módulo 5 no entrega un número fijo por área: entrega una **distribución** (media ± incertidumbre). Para llevar esa incertidumbre a las carreras:

1. Se toman **1 000 perfiles plausibles** del estudiante, sorteados del posterior $N(\mu_a, \Sigma_a)$ de cada área.
2. Para cada perfil se calcula la afinidad de todas las carreras y su orden.
3. **P(top 5)** = proporción de perfiles en los que la carrera quedó entre las 5 primeras.

Si la evidencia es sólida, el top 5 casi no cambia entre sorteos y las P(top 5) quedan cerca de 1. Si la evidencia es débil, el orden "baila" y las P(top 5) bajan. Ese es el aviso honesto de que conviene conversar más con el estudiante.

In [ ]:
# ── 7.1 Cargar y validar la tabla de carreras (reemplazable por país) ─────────
import unicodedata

RUTA_CARRERAS = f'{BASE}/03_referencias/carreras/carreras_EC.csv'

def slug(texto):
    t = unicodedata.normalize('NFKD', str(texto)).encode('ascii', 'ignore').decode().lower()
    return re.sub(r'[^a-z0-9]+', '-', t).strip('-')

def cargar_carreras(ruta):
    c = pd.read_csv(ruta, encoding='utf-8')
    c.columns = c.columns.str.strip()
    c = c.apply(lambda s: s.str.strip() if s.dtype == object else s)
    c = c.rename(columns={'Carrera universitaria': 'nombre', 'Universidad': 'universidad', 'Ciudad': 'ciudad'})
    c['id'] = c['nombre'].map(slug)
    c['id'] = c['id'] + c.groupby('id').cumcount().map(lambda k: '' if k == 0 else f'-{k + 1}')   # ids únicos
    hab = [col for col in c.columns if col.lower().startswith('habilidad')]
    c['habilidades'] = c[hab].apply(lambda r: [h for h in r if isinstance(h, str) and h], axis=1) if hab else [[]] * len(c)
    perfil = c[CODIGOS].apply(pd.to_numeric, errors='coerce')
    malas = perfil.isna().any(axis=1) | ~perfil.isin([1, 2, 3, 4, 5]).all(axis=1) | perfil.eq(1).all(axis=1)
    if malas.any():
        print(f'⚠️ {malas.sum()} carreras descartadas (perfil vacío, fuera de 1-5 o todo en 1):', c.loc[malas, 'nombre'].tolist())
    c, perfil = c[~malas].reset_index(drop=True), perfil[~malas].reset_index(drop=True)
    return c, perfil

carreras, PERFIL = cargar_carreras(RUTA_CARRERAS)
print(f'{len(carreras)} carreras válidas · columnas de perfil: {CODIGOS}')

cobertura = pd.DataFrame({'carreras que lo piden mucho (4-5)': PERFIL.ge(4).sum(), 'es el área principal': PERFIL.idxmax(axis=1).value_counts()}) \
    .reindex(CODIGOS).fillna(0).astype(int)
cobertura.index = [f'{a} · {EJES[a]}' for a in CODIGOS]
display(cobertura)
pocas = [a for a in CODIGOS if PERFIL[a].ge(4).sum() < 3]
if pocas:
    print(f'⚠️ Áreas con menos de 3 carreras que las pidan mucho: {pocas}. Un estudiante fuerte SOLO en esas áreas '
          'tendrá pocas opciones afines. Conviene completar la tabla (p. ej. Policía, Fuerzas Armadas, Gestión de Riesgos).')

In [ ]:
# ── 7.2 Motor de afinidad + Monte Carlo ──────────────────────────────────────
BETA_I = 0.50          # peso del interés (el rendimiento pesa 1 - BETA_I). Ajustable por el DECE.
N_SIM = 1000           # perfiles plausibles por estudiante
TOP_K = 5
N_EXPORTAR = 10        # carreras que se guardan por estudiante

def pesos_carrera(gamma):
    W = ((PERFIL[CODIGOS].values - 1) / 4) ** gamma
    return W / W.sum(axis=1, keepdims=True)

def posterior_matrices(ev, P, usar=('int', 'apt', 'notas')):
    # μ (7×2) y Σ (7×2×2) por estudiante, recalculados con el mismo motor del Módulo 5
    out = {}
    for id_est, g in ev.groupby('id'):
        g = g.set_index('area').reindex(CODIGOS)
        MU, SIG = np.zeros((7, 2)), np.zeros((7, 2, 2))
        for k, a in enumerate(CODIGOS):
            mu, Sig, _ = posterior_area({**g.loc[a].to_dict(), 'area': a}, P[a], 'aprendido', usar)
            MU[k], SIG[k] = mu, Sig
        out[id_est] = (MU, SIG)
    return out

def afinidad_cruda(MU, beta_I=BETA_I):
    # MU: (..., 7, 2) → promedio ponderado por el perfil de cada carrera (0-100)
    s = beta_I * norm.cdf(MU[..., 0]) + (1 - beta_I) * norm.cdf(MU[..., 1])
    return 100 * s @ WN_M.T

NORMAS_CARRERA = {}   # β → (media, desviación) de la afinidad cruda de cada carrera en la cohorte

def normas_carrera(beta_I=BETA_I):
    b = round(float(beta_I), 2)
    if b not in NORMAS_CARRERA:
        A = np.array([afinidad_cruda(MU, b) for MU, _ in MATRICES.values()])
        NORMAS_CARRERA[b] = (A.mean(axis=0), A.std(axis=0, ddof=1))
    return NORMAS_CARRERA[b]

def afinidad(MU, beta_I=BETA_I):
    # Afinidad equilibrada = percentil del estudiante en la cohorte para ESA carrera (0-100)
    m, sd = normas_carrera(beta_I)
    return 100 * norm.cdf((afinidad_cruda(MU, beta_I) - m) / sd)

def monte_carlo(MU, SIG, rng, beta_I=BETA_I, n=N_SIM, k=TOP_K):
    L = np.linalg.cholesky(SIG)                                        # (7,2,2)
    sorteos = MU + np.einsum('ajk,sak->saj', L, rng.standard_normal((n, 7, 2)))
    A = afinidad(sorteos, beta_I)                                       # (n, carreras)
    top = np.argpartition(-A, k, axis=1)[:, :k]
    p_top = np.bincount(top.ravel(), minlength=A.shape[1]) / n
    return p_top, np.percentile(A, 5, axis=0), np.percentile(A, 95, axis=0)

MATRICES = posterior_matrices(ev, PARAM)

def elegir_gamma(candidatos=(1, 1.5, 2, 3, 4), tv_max=0.10):
    # Equilibrio: el área de la carrera #1 debe repartirse como el área más fuerte de cada estudiante
    S = np.array([BETA_I * norm.cdf(MU[:, 0]) + (1 - BETA_I) * norm.cdf(MU[:, 1]) for MU, _ in MATRICES.values()])
    fuerte = np.bincount(S.argmax(1), minlength=7) / len(S)
    principal = PERFIL[CODIGOS].values.argmax(1)
    filas = []
    for g in candidatos:
        X = S @ pesos_carrera(g).T
        Z = (X - X.mean(0)) / X.std(0, ddof=1)
        uno = principal[Z.argmax(1)]
        filas.append({'γ': g, 'TV (0 = equilibrio perfecto)': 0.5 * np.abs(np.bincount(uno, minlength=7) / len(S) - fuerte).sum(),
                      'carrera #1 en su área más fuerte': np.mean(uno == S.argmax(1)),
                      'carreras distintas en algún top 3': len(set(np.argsort(-Z, 1)[:, :3].ravel())),
                      **{f'#1 en {a}': int((uno == k).sum()) for k, a in enumerate(CODIGOS)}})
    t = pd.DataFrame(filas).set_index('γ')
    ok = t.index[t['TV (0 = equilibrio perfecto)'] <= tv_max]
    return (float(ok[0]) if len(ok) else float(t['TV (0 = equilibrio perfecto)'].idxmin())), t

GAMMA_CARRERAS, tabla_gamma = elegir_gamma()
display(tabla_gamma.round(2))
print('Área más fuerte de los estudiantes:', dict(zip(CODIGOS, np.bincount(np.array([(BETA_I * norm.cdf(MU[:, 0]) + (1 - BETA_I) * norm.cdf(MU[:, 1])).argmax() for MU, _ in MATRICES.values()]), minlength=7).tolist())))
print(f'→ γ elegido = {GAMMA_CARRERAS}')
WN_M = pesos_carrera(GAMMA_CARRERAS)           # (carreras × 7)
normas_carrera(BETA_I)
rng = np.random.default_rng(11)
AFIN, PTOP, LO, HI = {}, {}, {}, {}
for id_est, (MU, SIG) in MATRICES.items():
    AFIN[id_est] = afinidad(MU)
    PTOP[id_est], LO[id_est], HI[id_est] = monte_carlo(MU, SIG, rng)
print(f'Afinidad y P(top {TOP_K}) calculadas para {len(AFIN)} estudiantes × {len(carreras)} carreras ({N_SIM} sorteos cada uno)')

### 7.C · Las razones: cada carrera explica por qué aparece

La afinidad es una suma de aportes por área, $\tilde w_{c,a}\,s_a$. La razón se arma con:

1. **El área que más aporta** a esa carrera, con su peso en la carrera (x/5), el percentil de interés y el de rendimiento.
2. **Las materias clave** del estudiante en esa área: las que más pesan en el área, con su nota del último año.
3. **Una advertencia** si interés y rendimiento no van juntos: "interés alto, conviene reforzar el rendimiento" o "rinde bien aunque declara poco interés". Es la misma lectura de los cuadrantes 🟡 y 🔵 del plano.

Así el DECE no recibe un número opaco, sino un argumento que puede discutir con el estudiante y su familia.

In [ ]:
# ── 7.3 Razones y ranking por estudiante ─────────────────────────────────────
ult = aporte.merge(por_anio.groupby('id')['grado'].max().rename('T'), on='id')
ult = ult[ult['grado'] == ult['T']]
notas_ult = por_materia.rename(columns={COL['materia']: 'materia'})[['id', 'grado', 'materia', 'nota10']]
ult = ult.merge(notas_ult, on=['id', 'grado', 'materia'], how='left')
MATERIAS_CLAVE = {k: g.sort_values(['w', 'nota10'], ascending=False).head(3)[['materia', 'nota10', 'w']].values.tolist()
                  for k, g in ult.groupby(['id', 'area'])}
PCT = post.set_index(['id', 'area'])[['pct_I', 'pct_R']]

def razon(id_est, j, beta_I=BETA_I):
    MU = MATRICES[id_est][0]
    s = beta_I * norm.cdf(MU[:, 0]) + (1 - beta_I) * norm.cdf(MU[:, 1])
    ap = WN_M[j] * s
    orden = np.argsort(-ap)
    a = CODIGOS[orden[0]]
    pI, pR = PCT.loc[(id_est, a)]
    txt = f'{EJES[a]} (pesa {int(PERFIL.loc[j, a])}/5 en la carrera): interés P{pI:.0f}, rendimiento P{pR:.0f}'
    mats = [f'{m} {n:.1f}' for m, n, _ in MATERIAS_CLAVE.get((id_est, a), []) if pd.notna(n)]
    if mats:
        txt += ' · materias: ' + ', '.join(mats)
    if pI >= 60 and pR < 40:
        txt += ' · interés alto, conviene reforzar el rendimiento'
    elif pR >= 60 and pI < 40:
        txt += ' · rinde bien aunque declara poco interés'
    a2 = CODIGOS[orden[1]]
    if ap[orden[1]] >= 0.5 * ap[orden[0]]:
        txt += f' · también suma {EJES[a2]}'
    return txt

filas = []
for id_est in AFIN:
    orden = np.argsort(-AFIN[id_est])[:N_EXPORTAR]
    for pos, j in enumerate(orden, 1):
        filas.append({'id': id_est, 'posicion': pos, 'carrera_id': carreras.loc[j, 'id'], 'carrera': carreras.loc[j, 'nombre'],
                      'afinidad': round(float(AFIN[id_est][j]), 1), 'p_top5': round(float(PTOP[id_est][j]), 2),
                      'rango90': [round(float(LO[id_est][j]), 1), round(float(HI[id_est][j]), 1)],
                      'universidad': carreras.loc[j, 'universidad'], 'ciudad': carreras.loc[j, 'ciudad'],
                      'razon': razon(id_est, j)})
ranking = pd.DataFrame(filas)

# Ejemplo: el estudiante de prueba con el perfil más definido (sin alertas de consistencia)
claros = [i for i in sorted(TEST) if ev.loc[ev['id'] == i, 'consistencia'].iloc[0] == 'sin_alertas']
ej = max(claros, key=lambda i: np.sort(PTOP[i])[-TOP_K:].mean())
print(f'▶ Estudiante {ej}: sus {N_EXPORTAR} carreras más afines (β_I = {BETA_I})\n')
for r in ranking[ranking['id'] == ej].itertuples():
    print(f'{r.posicion:2d}. {r.carrera:42s} afinidad {r.afinidad:5.1f} [{r.rango90[0]:.0f}-{r.rango90[1]:.0f}]  '
          f'P(top5) {r.p_top5:.2f}  · {r.universidad}, {r.ciudad}')
    print(f'    ↳ {r.razon}')

### Cómo leer el ejemplo

- **Afinidad** ordena. **[rango]** es el intervalo del 90 %: si dos carreras tienen rangos que se superponen mucho, el orden entre ellas no es firme.
- **P(top 5)** dice qué tan estable es esa posición. 0.95 = casi seguro entre las 5 primeras; 0.40 = depende de detalles que el modelo no conoce bien.
- La **razón** es el argumento para la conversación con el estudiante, no un veredicto.
- Muchas carreras de la tabla comparten perfil (por ejemplo, varias ingenierías). Por eso suelen aparecer **en bloque**. Eso es correcto: el sistema recomienda **familias** de carreras, y la elección dentro de la familia es parte de la orientación.

In [ ]:
# ── 7.4 Sensibilidad: ¿cuánto cambia el top 5 si el DECE mueve el 50/50? ──────
def top_k(A, k=TOP_K):
    return set(np.argsort(-A)[:k])

base = {i: top_k(AFIN[i]) for i in AFIN}
sens = {}
for b in [0.3, 0.5, 0.7]:
    tops = {i: top_k(afinidad(MATRICES[i][0], b)) for i in AFIN}
    sens[f'β_I={b:.1f} / β_R={1 - b:.1f}'] = {
        'top 5 en común con 50/50 (promedio de 5)': np.mean([len(tops[i] & base[i]) for i in AFIN]),
        '% con la misma carrera #1': np.mean([np.argmax(afinidad(MATRICES[i][0], b)) == np.argmax(AFIN[i]) for i in AFIN]) * 100}
tabla_sens = pd.DataFrame(sens).T
display(tabla_sens.round(2))
for b, r in tabla_sens.iterrows():
    if '0.5' in b: continue
    print(f"{b}: conserva {r.iloc[0]:.1f} de 5 carreras → " +
          ('robusto' if r.iloc[0] >= 4 else 'SENSIBLE: el peso interés/rendimiento cambia el top 5; conviene mostrarlo en la interfaz'))

### 7.D · Evaluación del ranking (solo con la simulación)

Con datos simulados conocemos el perfil **verdadero** de cada estudiante (`privado/`). Eso permite calcular su top 5 **verdadero** con la misma tabla de carreras, y comparar con el estimado. Se evalúa **solo con el 30 % de prueba**.

| Métrica | Pregunta | Ideal |
|---|---|---|
| Top 5 en común | ¿Cuántas de las 5 carreras estimadas están en el top 5 verdadero? | Cerca de 5 |
| #1 verdadera en el top 5 | ¿La carrera más afín de verdad aparece entre las 5 recomendadas? | Cerca de 100 % |
| Calibración de P(top 5) | Cuando el modelo dice 0.8, ¿la carrera está en el top 5 verdadero el 80 % de las veces? | Diagonal |
| Comparación sin notas | ¿Cuánto se pierde si solo se usa CHASIDE? | Debe perder: justifica incluir notas |

> ⚠️ Igual que en el Módulo 5: el rendimiento verdadero de la simulación nace de las notas, así que la ventaja de "con notas" está algo inflada. La **calibración** es la métrica más confiable de esta tabla.

In [ ]:
# ── 7.5 Evaluación del ranking contra la verdad simulada ─────────────────────
if os.path.exists(RUTA_VERDAD):
    V = verdad.set_index(['id', 'area'])
    sin_notas = posterior_matrices(ev[ev['id'].isin(TEST)], PARAM, usar=('int', 'apt'))
    res, cal = {'Bayes (notas + CHASIDE)': [], 'Solo CHASIDE (sin notas)': []}, []
    for i in sorted(TEST):
        MU_v = V.loc[i].reindex(CODIGOS)[['I_verdad', 'R_verdad']].values
        A_v = afinidad(MU_v)
        t_v = top_k(A_v)
        for nombre, MU in [('Bayes (notas + CHASIDE)', MATRICES[i][0]), ('Solo CHASIDE (sin notas)', sin_notas[i][0])]:
            t_e = top_k(afinidad(MU))
            res[nombre].append({'top 5 en común (de 5)': len(t_e & t_v), '#1 verdadera en el top 5': np.argmax(A_v) in t_e})
        cal += [(p, j in t_v) for j, p in enumerate(PTOP[i])]
    tabla_ranking = pd.DataFrame({k: pd.DataFrame(v).mean() for k, v in res.items()}).T
    display(tabla_ranking.round(2))
    cal = pd.DataFrame(cal, columns=['p', 'real'])
    cal = cal[cal['p'] > 0]
    cal['tramo'] = pd.cut(cal['p'], [0, .2, .4, .6, .8, 1.0], labels=['0-0.2', '0.2-0.4', '0.4-0.6', '0.6-0.8', '0.8-1'])
    print('Calibración de P(top 5): ¿cuando el modelo dice p, la carrera está en el top 5 verdadero con frecuencia p?')
    display(cal.groupby('tramo', observed=True).agg(n=('p', 'size'), p_dicha=('p', 'mean'), frecuencia_real=('real', 'mean')).round(2).T)
else:
    tabla_ranking = None
    print('Sin verdad simulada: la evaluación del ranking solo es posible con datos de prueba.')

In [ ]:
# ── 7.6 Guardar (una sola versión) ───────────────────────────────────────────
ranking.drop(columns='rango90').assign(afinidad_p5=ranking['rango90'].str[0], afinidad_p95=ranking['rango90'].str[1]) \
    .to_csv(f'{BASE}/05_salidas/ranking_carreras.csv', index=False)
if tabla_ranking is not None:
    tabla_ranking.to_csv(f'{BASE}/05_salidas/evaluacion_ranking.csv')
print('✓ 05_salidas/: ranking_carreras' + (' · evaluacion_ranking' if tabla_ranking is not None else ''))

### ✅ Aprobación Módulo 7
- [ ] Afinidad = promedio ponderado por el perfil de la carrera, con interés y rendimiento a **50/50** (ajustable).
- [ ] **P(top 5)** por Monte Carlo sobre el posterior: la incertidumbre del Módulo 5 llega hasta las carreras.
- [ ] Cada carrera trae su **razón**: área principal, materias clave y advertencia si interés y rendimiento no coinciden.
- [ ] Sensibilidad 30/70 · 50/50 · 70/30 y evaluación contra la verdad simulada.
- [ ] Revisar la tabla de carreras: completar las áreas con pocas carreras (ver aviso de 7.1).

---
# MÓDULO 9 · Exportación para la interfaz (JSON)

**Qué resuelve:** deja los resultados en archivos JSON que tu interfaz en JavaScript puede leer directamente con `fetch`, sin servidor ni base de datos.

```
05_salidas/web/
├── cohorte.json              ← índice: lista de estudiantes + resumen + evaluación
├── carreras.json             ← catálogo de carreras (perfil, habilidades, universidad)
└── estudiantes/
    └── E-xxxxxx.json         ← un archivo por estudiante (contrato abajo)
```

**Privacidad (son datos de menores):**

- El modelo trabaja solo con IDs seudónimos (`E-xxxxxx`). La llave ↔ ID **no sale** de `04_procesado/privado/`.
- **Nombres en la interfaz:** si existe `02_configuracion/nomina.csv` (columnas `id_sistema, nombre, genero`) se muestran los nombres reales; si no, cada estudiante recibe un **nombre ficticio estable** para poder conversar sobre "el caso de Camila" sin exponer a nadie. Con nombres reales, la carpeta `web/` debe protegerse con usuario y clave.
- No se publican enlaces abiertos. La carpeta `web/` se copia a la interfaz del DECE.
- **Una sola versión:** cada ejecución borra los JSON anteriores y escribe los nuevos.

### Contrato de `estudiantes/<id>.json`

| Clave | Contenido |
|---|---|
| `version`, `generado` | Versión del contrato y fecha de generación |
| `estudiante` | `id`, `grado_actual`, `fuentes` (notas: años y n.º de materias; chaside: fecha, modalidad, completitud, consistencia; ippj: `null` si no hay) |
| `areas[7]` | Por área: `interes` y `rendimiento` {percentil, rango90, z, sd}; `zona` {clave, texto, certeza, probabilidades}; `autopercepcion` {clave, texto}; `aportes`; `materias_clave` |
| `trayectoria[]` | Rendimiento por grado y área (percentil frente a su cohorte de ese grado) |
| `carreras[10]` | id, nombre, afinidad, `rango90`, p_top5, universidad, ciudad, razón |
| `alertas[]` | {tipo, texto}: avisos de calidad de la evidencia |
| `parametros` | Umbral de certeza y peso del interés usados |
| `areas[].evidencia` | Lo que entró al modelo en esa área: z del CHASIDE (interés y aptitud), z de las notas y peso de evidencia |

`cohorte.json` incluye `modelo`: claves del CHASIDE, normas de la cohorte, confiabilidades y parámetros aprendidos. Con eso la interfaz puede **recalcular al instante** a un estudiante que repite el CHASIDE, con las mismas fórmulas del Módulo 5.

En `cohorte.json`, cada estudiante trae además `perfil` = {área: [percentil interés, percentil rendimiento, zona]} para los gráficos de la cohorte sin abrir 500 archivos.

**Claves de zona:** `vocacion_consolidada` 🟢 · `interes_a_fortalecer` 🟡 · `talento_no_explorado` 🔵 · `no_prioritaria` ⚪ · `indeterminado` ❔.

In [ ]:
# ── 9.1 Funciones de exportación ─────────────────────────────────────────────
import shutil
from datetime import datetime

VERSION_CONTRATO = '1.0'
GENERADO = datetime.now().isoformat(timespec='seconds')
WEB = f'{BASE}/05_salidas/web'

CLAVE_ZONA = {CUADRANTES[(True, True)]: 'vocacion_consolidada', CUADRANTES[(True, False)]: 'interes_a_fortalecer',
              CUADRANTES[(False, True)]: 'talento_no_explorado', CUADRANTES[(False, False)]: 'no_prioritaria',
              '❔ Indeterminado: falta evidencia': 'indeterminado'}
CLAVE_AUTO = {'se percibe por ENCIMA de sus notas': 'por_encima', 'se percibe por DEBAJO de sus notas': 'por_debajo',
              'coherente con sus notas': 'coherente'}

def limpio(x):
    # Convierte tipos de numpy/pandas a JSON estándar (NaN → null)
    if isinstance(x, dict):
        return {k: limpio(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [limpio(v) for v in x]
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (float, np.floating)):
        return None if np.isnan(x) else round(float(x), 3)
    if isinstance(x, np.bool_):
        return bool(x)
    return x

def eje(mu, sd):
    return {'percentil': round(float(norm.cdf(mu) * 100)),
            'rango90': [round(float(norm.cdf(mu - 1.645 * sd) * 100)), round(float(norm.cdf(mu + 1.645 * sd) * 100))],
            'z': mu, 'sd': sd}

# Trayectoria: rendimiento de cada grado estandarizado contra la cohorte de ese grado y área
tray = por_anio[['id', 'grado', 'area', 'R']].copy()
tray['z'] = tray.groupby(['area', 'grado'])['R'].transform(lambda s: (s - s.mean()) / s.std())
tray['percentil'] = z_a_percentil(tray['z']).round(0)
TRAY = {i: g.sort_values(['grado', 'area']) for i, g in tray.groupby('id')}

FECHA_CH = ch.drop_duplicates('id').set_index('id')['fecha_aplicacion'] if 'fecha_aplicacion' in ch else pd.Series(dtype=str)
GRADOS = por_anio.groupby('id')['grado'].agg(lambda g: sorted(int(x) for x in g.unique()))
N_MAT = aporte.groupby('id')['materia'].nunique()
POST = {i: g.set_index('area').reindex(CODIGOS) for i, g in post.groupby('id')}
RANK = {i: g.sort_values('posicion') for i, g in ranking.groupby('id')}
EVQ = ev.drop_duplicates('id').set_index('id')
EVA = ev.set_index(['id', 'area'])

# ── Lo que la interfaz necesita para recalcular si se repite el CHASIDE ─────
# (mismas fórmulas del Módulo 5; la interfaz no "entrena", solo aplica estos valores)
disp_int = punt.xs('interes', axis=1, level=1).std(axis=1)
MODELO_WEB = {
    'claves': [[int(r.item), r.area, r.tipo] for r in claves.itertuples()],
    'normas_chaside': {a: {t: [float(punt[(a, t)].mean()), float(punt[(a, t)].std())] for t in ['interes', 'aptitud']} for a in CODIGOS},
    'consistencia': {'disp_p05': float(disp_int.quantile(0.05)), 'rango_si': [0.35, 0.65]},
    'alfa_chaside': {a: {t: ALFA_CH[(a, t)] for t in ['interes', 'aptitud']} for a in CODIGOS},
    'param': {a: {k: float(v) for k, v in PARAM[a].items()} for a in CODIGOS},
    'rubrica': {c: {n: f for n, (f, _) in niveles.items()} for c, niveles in RUBRICA.items()},
    'umbral_certeza': UMBRAL_CERTEZA, 'n_sim': N_SIM, 'top_k': TOP_K, 'beta': BETA_I,
}
# Texto de las 98 preguntas (opcional): 02_configuracion/chaside_preguntas.csv con columnas item, texto.
# Si existe, la interfaz muestra cada pregunta; si no, muestra una hoja de respuestas numerada del 1 al 98.
RUTA_PREGUNTAS = f'{BASE}/02_configuracion/chaside_preguntas.csv'
# Si no existe el archivo, se crea con preguntas de PROTOTIPO (redacción propia, misma estructura área/tipo de las claves).
# Para uso real, el DECE reemplaza la columna texto por el texto oficial y pone origen = oficial.
PREGUNTAS_PROTOTIPO = {
 ('C','interes'): [
  '¿Te gustaría administrar los recursos de una empresa o institución?',
  '¿Te interesaría llevar las cuentas y los presupuestos de un negocio?',
  '¿Te gustaría crear y dirigir tu propio emprendimiento?',
  '¿Disfrutarías organizar el trabajo de un equipo y asignar tareas?',
  '¿Te llama la atención entender cómo se mueven los precios y los mercados?',
  '¿Te gustaría trabajar en un banco, una cooperativa o una aseguradora?',
  '¿Te interesaría negociar acuerdos con proveedores o clientes?',
  '¿Te gustaría planificar cómo vender mejor un producto o servicio?',
  '¿Disfrutarías revisar que los registros y documentos estén en orden?',
  '¿Te atrae la idea de dirigir un área de una organización grande?'],
 ('C','aptitud'): [
  '¿Se te facilitan los cálculos con porcentajes, intereses o descuentos?',
  '¿Eres ordenado(a) para llevar registros y archivos?',
  '¿Sueles convencer a otros de apoyar una idea o un plan?',
  '¿Administras bien tu dinero y tu tiempo?'],
 ('H','interes'): [
  '¿Te gustaría investigar sobre la historia y las culturas de los pueblos?',
  '¿Te interesaría defender los derechos de las personas ante la ley?',
  '¿Disfrutarías escribir artículos, ensayos o crónicas?',
  '¿Te gustaría enseñar a niños o jóvenes?',
  '¿Te llama la atención entender por qué las sociedades actúan como actúan?',
  '¿Te gustaría trabajar en relaciones con otros países o con organismos internacionales?',
  '¿Disfrutarías debatir sobre temas sociales y políticos?',
  '¿Te interesaría aprender varios idiomas y trabajar con ellos?',
  '¿Te gustaría acompañar a personas o comunidades a resolver sus problemas?',
  '¿Te atrae leer filosofía o reflexionar sobre preguntas profundas?'],
 ('H','aptitud'): [
  '¿Se te facilita expresar tus ideas por escrito con claridad?',
  '¿Recuerdas con facilidad lo que lees?',
  '¿Sabes argumentar y sostener una postura en una discusión?',
  '¿Aprendes idiomas con facilidad?'],
 ('A','interes'): [
  '¿Te gustaría dibujar, pintar o ilustrar como parte de tu trabajo?',
  '¿Disfrutarías diseñar espacios, objetos o ropa?',
  '¿Te interesaría componer, tocar o producir música?',
  '¿Te gustaría actuar o dirigir obras de teatro?',
  '¿Te llama la atención crear videos, animaciones o fotografías?',
  '¿Disfrutarías diseñar la imagen visual de una marca?',
  '¿Te gustaría visitar museos y conocer el trabajo de artistas?',
  '¿Te interesaría crear personajes o mundos para un videojuego o una historia?',
  '¿Disfrutarías trabajar con tus manos para crear piezas únicas?',
  '¿Te gustaría dedicarte a una actividad donde la creatividad sea lo principal?'],
 ('A','aptitud'): [
  '¿Tienes facilidad para dibujar o representar ideas visualmente?',
  '¿Tienes buen oído para el ritmo y la música?',
  '¿Notas con facilidad si los colores y las formas combinan bien?',
  '¿Te expresas bien con el cuerpo o la voz frente a otros?'],
 ('S','interes'): [
  '¿Te gustaría atender a personas enfermas o heridas?',
  '¿Te interesaría saber cómo funciona el cuerpo humano por dentro?',
  '¿Disfrutarías ayudar a alguien a recuperar su movilidad o su salud?',
  '¿Te gustaría trabajar en un hospital, una clínica o un centro de salud?',
  '¿Te interesaría la alimentación saludable y su efecto en el cuerpo?',
  '¿Te gustaría cuidar la salud de los animales?',
  '¿Disfrutarías escuchar y apoyar a personas con dificultades emocionales?',
  '¿Te interesaría investigar medicamentos y cómo actúan?',
  '¿Te gustaría trabajar en campañas de prevención de enfermedades?',
  '¿Te atrae la idea de atender emergencias médicas?'],
 ('S','aptitud'): [
  '¿Mantienes la calma ante la sangre o las heridas?',
  '¿Tienes paciencia para cuidar a otras personas?',
  '¿Se te facilita la biología?',
  '¿Tienes precisión y buen pulso con las manos?'],
 ('I','interes'): [
  '¿Te gustaría diseñar y construir máquinas o robots?',
  '¿Te interesaría programar aplicaciones o sistemas informáticos?',
  '¿Disfrutarías entender cómo funcionan los circuitos y la electricidad?',
  '¿Te gustaría diseñar puentes, edificios o carreteras?',
  '¿Te interesaría mejorar los procesos de una fábrica?',
  '¿Te gustaría trabajar con redes, servidores o ciberseguridad?',
  '¿Disfrutarías arreglar motores, aparatos o equipos?',
  '¿Te interesaría analizar grandes cantidades de datos con computadoras?',
  '¿Te gustaría crear soluciones técnicas para problemas cotidianos?',
  '¿Te atrae trabajar con planos, maquetas o modelos en 3D?'],
 ('I','aptitud'): [
  '¿Se te facilita entender cómo están armadas las cosas?',
  '¿Resuelves problemas de lógica con facilidad?',
  '¿Aprendes rápido a usar programas o herramientas nuevas?',
  '¿Imaginas bien los objetos en tres dimensiones?'],
 ('D','interes'): [
  '¿Te gustaría proteger a tu comunidad ante emergencias o desastres?',
  '¿Te interesaría formarte en una escuela militar o policial?',
  '¿Disfrutarías trabajar con disciplina y orden jerárquico?',
  '¿Te gustaría investigar delitos y buscar evidencias?',
  '¿Te interesaría defender la soberanía y las fronteras del país?',
  '¿Te gustaría coordinar planes de evacuación o de rescate?',
  '¿Disfrutarías actividades que exigen buena condición física?',
  '¿Te interesaría trabajar en la seguridad de personas o instalaciones?',
  '¿Te gustaría servir a tu país en situaciones de riesgo?',
  '¿Te atrae la idea de usar uniforme y formar parte de una institución de servicio?'],
 ('D','aptitud'): [
  '¿Mantienes la calma y decides rápido en situaciones de peligro?',
  '¿Cumples normas e instrucciones con rigor?',
  '¿Tienes buena resistencia física?',
  '¿Sabes liderar a un grupo en una situación difícil?'],
 ('E','interes'): [
  '¿Te gustaría investigar en un laboratorio?',
  '¿Te interesaría estudiar los fenómenos de la naturaleza y el universo?',
  '¿Disfrutarías resolver problemas matemáticos complejos?',
  '¿Te gustaría trabajar en el campo con cultivos o animales?',
  '¿Te interesaría estudiar las plantas, los animales y los ecosistemas?',
  '¿Te gustaría descubrir nuevos materiales o sustancias químicas?',
  '¿Disfrutarías hacer experimentos para comprobar una idea?',
  '¿Te interesaría cuidar el ambiente y los recursos naturales?',
  '¿Te gustaría estudiar la Tierra, los volcanes o los suelos?',
  '¿Te atrae la idea de dedicarte a la ciencia y publicar descubrimientos?'],
 ('E','aptitud'): [
  '¿Se te facilitan las matemáticas?',
  '¿Eres observador(a) y meticuloso(a) al hacer un experimento?',
  '¿Entiendes con facilidad las fórmulas de física o química?',
  '¿Tienes paciencia para trabajos que requieren repetir y comprobar?'],
}
if not os.path.exists(RUTA_PREGUNTAS):
    usado = {k: 0 for k in PREGUNTAS_PROTOTIPO}
    filas = []
    for r in claves.sort_values('item').itertuples():
        k = (r.area, r.tipo); filas.append({'item': int(r.item), 'texto': PREGUNTAS_PROTOTIPO[k][usado[k]], 'origen': 'prototipo'}); usado[k] += 1
    pd.DataFrame(filas).to_csv(RUTA_PREGUNTAS, index=False, encoding='utf-8')
    print('✓ 02_configuracion/chaside_preguntas.csv creado con preguntas de prototipo')
if os.path.exists(RUTA_PREGUNTAS):
    pq = pd.read_csv(RUTA_PREGUNTAS, dtype={'item': int, 'texto': str})
    MODELO_WEB['preguntas'] = {int(r.item): r.texto for r in pq.itertuples()}
    MODELO_WEB['preguntas_origen'] = 'prototipo' if 'origen' in pq and (pq['origen'] == 'prototipo').any() else 'oficial'


# ── Nombres para la interfaz ─────────────────────────────────────────────────
# Si existe 02_configuracion/nomina.csv (id_sistema, nombre, genero) se usan los nombres reales.
# Si no, cada ID recibe un nombre FICTICIO estable (mismo ID → mismo nombre en cada ejecución).
import hashlib
NOMBRES_F = ['Camila', 'Valentina', 'Isabella', 'Sofía', 'Daniela', 'Mariana', 'Gabriela', 'Emilia', 'Paula', 'Andrea',
             'Doménica', 'Ariana', 'Nicole', 'Fernanda', 'Melanie', 'Antonella', 'Ana Paula', 'Juliana', 'Renata', 'Victoria',
             'Carolina', 'Alejandra', 'Micaela', 'Dayana', 'Paulina', 'Rafaela', 'Sara', 'Elena', 'Martina', 'Josefina']
NOMBRES_M = ['Mateo', 'Santiago', 'Sebastián', 'Nicolás', 'Martín', 'Daniel', 'Alejandro', 'Joaquín', 'Diego', 'Gabriel',
             'Emilio', 'Juan Pablo', 'Andrés', 'David', 'Samuel', 'Tomás', 'Benjamín', 'Felipe', 'Esteban', 'Leonardo',
             'Carlos', 'José', 'Adrián', 'Ricardo', 'Matías', 'Francisco', 'Pedro', 'Bruno', 'Iván', 'Josué']
APELLIDOS = ['Andrade', 'Vásconez', 'Salazar', 'Cevallos', 'Paredes', 'Guerrero', 'Rivadeneira', 'Zambrano', 'Mora', 'Espinosa',
             'Benítez', 'Villacís', 'Jaramillo', 'Proaño', 'Carrión', 'Ortiz', 'Chávez', 'Molina', 'Rivas', 'Torres',
             'Naranjo', 'Erazo', 'Pazmiño', 'Montalvo', 'Cordero', 'Aguirre', 'Bustamante', 'Calderón', 'Escobar', 'Flores',
             'Herrera', 'León', 'Maldonado', 'Navarrete', 'Ponce', 'Quintana', 'Rosero', 'Suárez', 'Tapia', 'Unda',
             'Valencia', 'Yépez', 'Arias', 'Burbano', 'Coronel', 'Dávila', 'Endara', 'Freire', 'Garcés', 'Hidalgo']
SISTEMA = dict(zip(llave['id'], llave['nombre'].astype(str)))
RUTA_NOMINA = f'{BASE}/02_configuracion/nomina.csv'
NOMINA = {}
if os.path.exists(RUTA_NOMINA):
    nom = pd.read_csv(RUTA_NOMINA, dtype=str)
    NOMINA = {r.id_sistema: (r.nombre, (r.genero or '').upper()[:1] or None) for r in nom.itertuples()}

def nombre_de(id_est):
    real = NOMINA.get(SISTEMA.get(id_est))
    if real:
        return real[0], real[1], False
    h = int(hashlib.sha256(id_est.encode()).hexdigest(), 16)
    genero = 'F' if h % 2 else 'M'
    nombres = NOMBRES_F if genero == 'F' else NOMBRES_M
    return f'{nombres[(h >> 8) % len(nombres)]} {APELLIDOS[(h >> 20) % len(APELLIDOS)]} {APELLIDOS[(h >> 40) % len(APELLIDOS)]}', genero, True

# ── Datos de origen para la sección "Fuente de datos" ────────────────────────
GRADO_ACTUAL = notas.groupby('id')['Grade Level'].apply(lambda g: pd.to_numeric(g, errors='coerce').max()) if 'Grade Level' in notas else pd.Series(dtype=float)
src = notas[['id', 'grado', COL['anio'], COL['materia'], 'nota10', 'estado', 'regla']].copy()
src['periodo'] = src[COL['anio']].astype(str).str.extract(r'(\d{4}\s*-\s*\d{4})')[0]
src['sem'] = src[COL['anio']].astype(str).str.extract(r'Sem\s*(\d)')[0]
AREAS_REGLA = {i: ''.join(a for a in CODIGOS if W.loc[i, a] > 0) for i in W.index}
SRC = dict(tuple(src.groupby('id')))
ITEMS_AREA = {(a, t): [f'P{i}' for i in claves.query('area == @a and tipo == @t')['item']] for a in CODIGOS for t in ['interes', 'aptitud']}

def notas_origen(id_est):
    g = SRC.get(id_est)
    if g is None:
        return []
    anios = []
    for (gr, per), h in g.groupby(['grado', 'periodo'], dropna=False):
        mats = []
        for m, k in h.groupby(COL['materia']):
            regla = k['regla'].iloc[0]
            mats.append({'materia': m, 'sem1': k.loc[k['sem'] == '1', 'nota10'].mean(), 'sem2': k.loc[k['sem'] == '2', 'nota10'].mean(),
                         'final': k['nota10'].mean(), 'estado': k['estado'].iloc[0],
                         'areas': AREAS_REGLA.get(regla, '') if pd.notna(regla) else ''})
        orden = {'incluida': 0, 'sin_regla': 1, 'excluida': 2}
        anios.append({'grado': int(gr), 'periodo': per if isinstance(per, str) else None,
                      'materias': sorted(mats, key=lambda x: (orden.get(x['estado'], 3), x['materia']))})
    return sorted(anios, key=lambda x: x['grado'])

def chaside_origen(id_est):
    if id_est not in resp.index:
        return None
    r = resp.loc[id_est]
    return {'respondidas': int(r.notna().sum()), 'total': int(len(r)), 'si': int((r == 1).sum()),
            'areas': {a: {t: [int((r[ITEMS_AREA[(a, t)]] == 1).sum()), len(ITEMS_AREA[(a, t)])] for t in ['interes', 'aptitud']} for a in CODIGOS}}

def alertas(id_est, areas):
    q, out = EVQ.loc[id_est], []
    if q['consistencia'] == 'alerta':
        out.append({'tipo': 'consistencia', 'texto': 'CHASIDE con respuestas poco diferenciadas: confirmar intereses en entrevista.'})
    if q['completitud'] != 'completa':
        out.append({'tipo': 'completitud', 'texto': f"CHASIDE {q['completitud']} ({q['completitud_%']:.0f} % respondido)."})
    if q['antiguedad'] != 'menos_1_anio':
        out.append({'tipo': 'antiguedad', 'texto': 'CHASIDE aplicado hace más de un año: los intereses pueden haber cambiado.'})
    if q['modalidad'] == 'autonoma':
        out.append({'tipo': 'modalidad', 'texto': 'CHASIDE respondido sin supervisión: su confiabilidad se ajustó a la baja.'})
    g = GRADOS.get(id_est, [])
    if len(g) < 2:
        out.append({'tipo': 'notas', 'texto': f'Solo hay notas de {len(g)} año: el rendimiento tiene menos respaldo.'})
    n_ind = sum(a['zona']['clave'] == 'indeterminado' for a in areas)
    if n_ind >= 3:
        out.append({'tipo': 'indeterminado', 'texto': f'{n_ind} de 7 áreas sin zona clara: falta evidencia (sumar entrevista u otra prueba).'})
    return out

def json_estudiante(id_est):
    p = POST[id_est]
    areas = []
    for a, r in p.iterrows():
        probs = {CLAVE_ZONA[z]: r[f'P[{z}]'] for z in CUADRANTES.values()}
        auto = r.get('autopercepcion')
        areas.append({
            'codigo': a, 'nombre': EJES[a],
            'interes': eje(r['mu_I'], r['sd_I']), 'rendimiento': eje(r['mu_R'], r['sd_R']),
            'zona': {'clave': CLAVE_ZONA[r['cuadrante']], 'texto': r['cuadrante'][2:], 'certeza': r['certeza'], 'probabilidades': probs},
            'autopercepcion': {'clave': CLAVE_AUTO.get(auto), 'texto': auto} if isinstance(auto, str) else None,
            'aportes': {'interes': {'chaside_interes': r.get('aporte_I · CHASIDE interés')},
                        'rendimiento': {'notas': r.get('aporte_R · Notas'), 'autopercepcion': r.get('aporte_R · Autopercepción')}},
            'materias_clave': [{'materia': m, 'nota': n, 'peso': w} for m, n, w in MATERIAS_CLAVE.get((id_est, a), [])],
            'evidencia': {k: EVA.loc[(id_est, a), k] for k in ['z_int', 'z_apt', 'z_notas', 'W_efectiva']}})
    q = EVQ.loc[id_est]
    nombre, genero, ficticio = nombre_de(id_est)
    ga = GRADO_ACTUAL.get(id_est)
    return limpio({
        'version': VERSION_CONTRATO, 'generado': GENERADO,
        'estudiante': {'id': id_est, 'nombre': nombre, 'genero': genero, 'nombre_ficticio': ficticio,
                       'grado_actual': int(ga) if pd.notna(ga) else max(GRADOS.get(id_est, [None]) or [None]),
                       'fuentes': {'notas': {'anios': GRADOS.get(id_est, []), 'materias': N_MAT.get(id_est, 0)},
                                   'chaside': {'fecha': FECHA_CH.get(id_est), 'modalidad': q['modalidad'],
                                               'completitud': q['completitud'], 'consistencia': q['consistencia']},
                                   'ippj': None}},
        'areas': areas,
        'trayectoria': [{'grado': int(t.grado), 'area': t.area, 'rendimiento_percentil': t.percentil}
                        for t in TRAY.get(id_est, pd.DataFrame()).itertuples()],
        'carreras': [{'id': c.carrera_id, 'nombre': c.carrera, 'afinidad': c.afinidad, 'rango90': c.rango90, 'p_top5': c.p_top5,
                      'universidad': c.universidad, 'ciudad': c.ciudad, 'razon': c.razon} for c in RANK[id_est].itertuples()],
        'alertas': alertas(id_est, areas),
        'origen': {'notas': notas_origen(id_est), 'chaside': chaside_origen(id_est)},
        'parametros': {'umbral_certeza': UMBRAL_CERTEZA, 'peso_interes_carreras': BETA_I, 'simulaciones_monte_carlo': N_SIM}})

In [ ]:
# ── 9.2 Escribir los archivos (una sola versión) ─────────────────────────────
if os.path.exists(WEB):
    shutil.rmtree(WEB)
os.makedirs(f'{WEB}/estudiantes')

def escribir(ruta, obj):
    with open(ruta, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, separators=(',', ':'))   # compacto: carga más rápido en la web

indice = []
for id_est in sorted(AFIN):
    d = json_estudiante(id_est)
    escribir(f'{WEB}/estudiantes/{id_est}.json', d)
    indice.append({'id': id_est, 'nombre': d['estudiante']['nombre'], 'genero': d['estudiante']['genero'],
                   'nombre_ficticio': d['estudiante']['nombre_ficticio'], 'grado_actual': d['estudiante']['grado_actual'],
                   'top3': [c['nombre'] for c in d['carreras'][:3]],
                   'vocacion_consolidada': [a['codigo'] for a in d['areas'] if a['zona']['clave'] == 'vocacion_consolidada'],
                   'perfil': {a['codigo']: [a['interes']['percentil'], a['rendimiento']['percentil'], a['zona']['clave']] for a in d['areas']},
                   'n_alertas': len(d['alertas'])})

escribir(f'{WEB}/carreras.json', limpio({
    'version': VERSION_CONTRATO, 'generado': GENERADO, 'pais': 'EC', 'fuente': 'carreras_EC.csv',
    'gamma': GAMMA_CARRERAS,
    'normas': {f'{b:.1f}': {'media': normas_carrera(b)[0].tolist(), 'sd': normas_carrera(b)[1].tolist()} for b in np.round(np.arange(0, 1.01, 0.1), 1)},
    'areas': [{'codigo': a, 'nombre': EJES[a]} for a in CODIGOS],
    'carreras': [{'id': r['id'], 'nombre': r['nombre'], 'perfil': {a: int(PERFIL.loc[j, a]) for a in CODIGOS},
                  'habilidades': r['habilidades'], 'universidad': r['universidad'], 'ciudad': r['ciudad']}
                 for j, r in carreras.iterrows()]}))

zonas = post['cuadrante'].map(CLAVE_ZONA).value_counts(normalize=True).round(3).to_dict()
evaluacion = None
if 'tabla_eval' in globals():
    evaluacion = {'origen': 'simulación (Módulo 3B) · 30 % de prueba',
                  'perfil': tabla_eval.loc['Bayes aprendido (núcleo completo)'].to_dict(),
                  'ranking': tabla_ranking.loc['Bayes (notas + CHASIDE)'].to_dict() if tabla_ranking is not None else None}
escribir(f'{WEB}/cohorte.json', limpio({
    'version': VERSION_CONTRATO, 'generado': GENERADO, 'n_estudiantes': len(indice),
    'parametros': {'umbral_certeza': UMBRAL_CERTEZA, 'peso_interes_carreras': BETA_I},
    'resumen_zonas': zonas, 'evaluacion': evaluacion, 'modelo': MODELO_WEB, 'estudiantes': indice}))

tam = sum(os.path.getsize(f'{WEB}/estudiantes/{f}') for f in os.listdir(f'{WEB}/estudiantes'))
print(f'✓ 05_salidas/web/: cohorte.json · carreras.json · {len(indice)} archivos en estudiantes/ ({tam / 1e6:.1f} MB en total)')

In [ ]:
# ── 9.3 Vista previa: un estudiante de ejemplo (recortado) ───────────────────
d = json.load(open(f'{WEB}/estudiantes/{ej}.json', encoding='utf-8'))
corto = {**d, 'areas': d['areas'][:1], 'trayectoria': d['trayectoria'][:3], 'carreras': d['carreras'][:2]}
print(json.dumps(corto, ensure_ascii=False, indent=1))

### 9.4 · Conexión desde tu interfaz (JavaScript vanilla)

Copia la carpeta `05_salidas/web/` dentro de tu proyecto como `datos/`:

```javascript
const COLOR_ZONA = {
  vocacion_consolidada: '#2e9e5b', interes_a_fortalecer: '#e0b100',
  talento_no_explorado: '#2f6fd6', no_prioritaria: '#b0b0b0', indeterminado: '#ffffff'
};

async function cargarEstudiante(id) {
  const r = await fetch(`datos/estudiantes/${id}.json`);
  if (!r.ok) throw new Error(`No existe ${id}`);
  return r.json();
}

// Índice para el selector de estudiantes
const cohorte = await (await fetch('datos/cohorte.json')).json();
const est = await cargarEstudiante(cohorte.estudiantes[0].id);

// Plano cartesiano: x = rendimiento, y = interés
est.areas.forEach(a => dibujarPunto(a.rendimiento.percentil, a.interes.percentil, COLOR_ZONA[a.zona.clave], a.codigo));
// Carreras
est.carreras.forEach(c => console.log(c.nombre, c.afinidad, c.p_top5, c.razon));
```

> `fetch` necesita que la página se abra desde un servidor (por ejemplo, la extensión *Live Server* o `python -m http.server`), no con doble clic sobre el archivo.

### ✅ Aprobación Módulo 9
- [ ] Estructura `web/` con `cohorte.json`, `carreras.json` y un JSON por estudiante.
- [ ] Contrato de campos (tabla de arriba) y claves de zona.
- [ ] Privacidad: solo IDs seudónimos, una sola versión, sin enlaces públicos.